In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install segmentation-models-pytorch --quiet

import segmentation_models_pytorch as smp

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

In [ ]:
"""
Spine Curve Detection - Train model để detect đường cong spine từ landmarks
Approach: Landmarks -> Smooth Curve -> Segmentation/Distance Transform
"""
import sys
sys.path.append("/content/drive/MyDrive/Colab Notebooks")

from CONFIG import CONFIG
import numpy as np
import cv2
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from pycocotools.coco import COCO
from scipy.interpolate import splprep, splev, UnivariateSpline
from scipy.ndimage import distance_transform_edt
import matplotlib.pyplot as plt
import warnings
import random
from scipy.interpolate import splprep, splev
from google.colab.patches import cv2_imshow
from scipy.signal import savgol_filter
from skimage.morphology import skeletonize
import string
warnings.filterwarnings('ignore')

from scipy.ndimage import convolve
from collections import deque


In [ ]:
# @title
import os
import json
import cv2
import numpy as np
from typing import List, Tuple, Union, Optional, Dict, Any


class KeypointSaver:
    """
    A flexible keypoint annotation saver that supports unlimited keypoints per image.
    Saves images and their keypoint annotations in COCO format.
    """

    def __init__(self, output_dir: str = "output", annotation_file: str = "_annotations.json"):
        """
        Initialize the KeypointSaver.

        Args:
            output_dir: Directory to save images and annotations
            annotation_file: Name of the JSON file to store keypoint annotations
        """
        self.output_dir = output_dir
        self.annotation_file = annotation_file
        self.annotation_path = os.path.join(output_dir, annotation_file)

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Load existing annotations or create new
        self.annotations = self._load_annotations()

    def _load_annotations(self) -> Dict[str, Any]:
        """
        Load annotations from file if exists, otherwise create new structure.

        Returns:
            Dictionary containing images, annotations, and categories
        """
        if os.path.exists(self.annotation_path):
            try:
                with open(self.annotation_path, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                print(f"⚠ Warning: Could not parse {self.annotation_path}, creating new file")
                return self._create_empty_annotations()
        else:
            return self._create_empty_annotations()

    def _create_empty_annotations(self) -> Dict[str, Any]:
        """
        Create empty annotations structure.

        Returns:
            Empty COCO-format annotation dictionary
        """
        return {
            "images": [],
            "annotations": [],
            "categories": [{
                "id": 1,
                "name": "object",
                "supercategory": "none",
                "keypoints": [],  # Will be dynamically populated
                "skeleton": []
            }]
        }

    def _save_annotations(self) -> None:
        """Save annotations to JSON file."""
        with open(self.annotation_path, 'w', encoding='utf-8') as f:
            json.dump(self.annotations, f, indent=2, ensure_ascii=False)

    def _normalize_keypoints(self, keypoints: List[Union[List[float], Tuple[float, ...]]]) -> List[List[float]]:
        """
        Normalize keypoints to ensure they all have visibility values.

        Args:
            keypoints: List of keypoints in format [[x1,y1], [x2,y2], ...] or [[x1,y1,v1], [x2,y2,v2], ...]

        Returns:
            Normalized keypoints with visibility: [[x1,y1,v1], [x2,y2,v2], ...]
            Visibility values: 0=not labeled, 1=labeled but not visible, 2=labeled and visible
        """
        normalized = []
        for kp in keypoints:
            # Convert to list if numpy array
            if isinstance(kp, np.ndarray):
                kp = kp.tolist()

            kp_len = len(kp)
            if kp_len == 2:  # [x, y] -> add default visibility=2 (visible)
                normalized.append([float(kp[0]), float(kp[1]), 2.0])
            elif kp_len == 3:  # [x, y, v] -> keep as is
                normalized.append([float(kp[0]), float(kp[1]), float(kp[2])])
            else:
                raise ValueError(f"Invalid keypoint format: {kp}. Expected [x,y] or [x,y,v]")
        return normalized

    def _calculate_bbox(self, keypoints: List[List[float]], padding: float = 5.0) -> List[float]:
        """
        Calculate bounding box from keypoints with optional padding.

        Args:
            keypoints: Normalized keypoints [[x1,y1,v1], [x2,y2,v2], ...]
            padding: Padding to add around keypoints (in pixels)

        Returns:
            Bounding box in format [x, y, width, height]
        """
        # Extract visible keypoints only
        visible_kps = [kp for kp in keypoints if kp[2] > 0]

        if visible_kps is None or len(visible_kps) == 0:
            # If no visible keypoints, use all keypoints
            visible_kps = keypoints

        xs = [kp[0] for kp in visible_kps]
        ys = [kp[1] for kp in visible_kps]

        x_min = max(0, min(xs) - padding)
        y_min = max(0, min(ys) - padding)
        x_max = max(xs) + padding
        y_max = max(ys) + padding

        width = x_max - x_min
        height = y_max - y_min

        return [x_min, y_min, width, height]

    def _update_category_keypoints(self, num_keypoints: int) -> None:
        """
        Update category keypoints list to match the number of keypoints.

        Args:
            num_keypoints: Number of keypoints in the annotation
        """
        if self.annotations["categories"]:
            category = self.annotations["categories"][0]
            current_kp_names = category.get("keypoints", [])

            # If current list is shorter, extend it
            if len(current_kp_names) < num_keypoints:
                for i in range(len(current_kp_names), num_keypoints):
                    current_kp_names.append(f"point_{i+1}")
                category["keypoints"] = current_kp_names

    def save_image_with_keypoints(
        self,
        image: Union[str, np.ndarray],
        keypoints: List[Union[List[float], Tuple[float, ...], np.ndarray]],
        image_name: Optional[str] = None,
        category_id: int = 1,
        bbox_padding: float = 5.0
    ) -> Optional[Dict[str, Any]]:
        """
        Save image with keypoints to output directory and update annotations file.

        Args:
            image: Image as numpy array (BGR format) or path to image file
            keypoints: List of keypoints [[x1,y1], [x2,y2], ...] or [[x1,y1,v1], [x2,y2,v2], ...]
            image_name: Custom image filename (auto-generated if None)
            category_id: Category ID for the annotation (default: 1)
            bbox_padding: Padding around keypoints for bounding box calculation

        Returns:
            Dictionary containing image info, annotation, and image path, or None if failed
        """
        # Load image if path is provided
        if isinstance(image, str):
            loaded_image = cv2.imread(image)
            if loaded_image is None:
                print(f"✗ Error: Could not read image from path: {image}")
                return None
            image = loaded_image

        # Validate image
        if not isinstance(image, np.ndarray):
            print(f"✗ Error: Invalid image type: {type(image)}")
            return None

        # Check if image is empty
        if image.size == 0:
            print("✗ Error: Image array is empty")
            return None

        # Validate keypoints
        if keypoints is None or len(keypoints) == 0:

            print("✗ Error: No keypoints provided")
            return None

        try:
            # Normalize keypoints
            keypoints_normalized = self._normalize_keypoints(keypoints)
        except ValueError as e:
            print(f"✗ Error: {e}")
            return None
        except Exception as e:
            print(f"✗ Error normalizing keypoints: {e}")
            return None

        # Generate image name if not provided
        if image_name is None:
            img_count = len(self.annotations["images"]) + 1
            image_name = f"image_{img_count:06d}.jpg"

        # Ensure file extension
        if not image_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            image_name += '.jpg'

        # Full image path
        image_path = os.path.join(self.output_dir, image_name)

        # Save image
        try:
            success = cv2.imwrite(image_path, image)
            if not success:
                print(f"✗ Error: Could not save image to {image_path}")
                return None
        except Exception as e:
            print(f"✗ Error saving image: {e}")
            return None

        # Get image dimensions
        height, width = image.shape[:2]

        # Create image info
        image_id = len(self.annotations["images"]) + 1
        image_info = {
            "id": image_id,
            "file_name": image_name,
            "width": int(width),
            "height": int(height)
        }
        self.annotations["images"].append(image_info)

        # Create annotation
        annotation_id = len(self.annotations["annotations"]) + 1

        # Flatten keypoints to COCO format: [x1,y1,v1,x2,y2,v2,...]
        keypoints_flat = []
        for kp in keypoints_normalized:
            keypoints_flat.extend(kp)

        # Calculate bounding box
        bbox = self._calculate_bbox(keypoints_normalized, bbox_padding)

        # Count visible keypoints
        num_visible = sum(1 for kp in keypoints_normalized if kp[2] > 0)

        annotation = {
            "id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "keypoints": keypoints_flat,
            "num_keypoints": num_visible,
            "bbox": bbox,
            "area": bbox[2] * bbox[3],  # width * height
            "iscrowd": 0
        }
        self.annotations["annotations"].append(annotation)

        # Update category keypoints if needed
        self._update_category_keypoints(len(keypoints_normalized))

        # Save annotations to file
        self._save_annotations()

        print(f"✅ Saved: {image_path}")
        print(f"✅ Added {len(keypoints_normalized)} keypoints ({num_visible} visible)")

        return {
            "image_info": image_info,
            "annotation": annotation,
            "image_path": image_path
        }

    def get_stats(self) -> Dict[str, Any]:
        """
        Get statistics about saved annotations.

        Returns:
            Dictionary containing statistics
        """
        total_images = len(self.annotations['images'])
        total_annotations = len(self.annotations['annotations'])

        # Calculate keypoint statistics
        if total_annotations > 0:
            keypoint_counts = [
                len(ann['keypoints']) // 3
                for ann in self.annotations['annotations']
            ]
            min_kp = min(keypoint_counts)
            max_kp = max(keypoint_counts)
            avg_kp = sum(keypoint_counts) / len(keypoint_counts)
        else:
            min_kp = max_kp = avg_kp = 0

        stats = {
            "total_images": total_images,
            "total_annotations": total_annotations,
            "min_keypoints": min_kp,
            "max_keypoints": max_kp,
            "avg_keypoints": avg_kp,
            "annotation_file": self.annotation_path
        }

        print("\n" + "=" * 60)
        print("📊 ANNOTATION STATISTICS")
        print("=" * 60)
        print(f"Total images:       {stats['total_images']}")
        print(f"Total annotations:  {stats['total_annotations']}")
        print(f"Keypoints per annotation:")
        print(f"  - Minimum:        {stats['min_keypoints']}")
        print(f"  - Maximum:        {stats['max_keypoints']}")
        print(f"  - Average:        {stats['avg_keypoints']:.2f}")
        print(f"Annotation file:    {stats['annotation_file']}")
        print("=" * 60 + "\n")

        return stats

    def clear_all(self, confirm: bool = False) -> None:
        """
        Clear all annotations (does not delete image files).

        Args:
            confirm: Set to True to confirm deletion (safety measure)
        """
        if not confirm:
            print("⚠ Warning: Set confirm=True to clear all annotations")
            return

        self.annotations = self._create_empty_annotations()
        self._save_annotations()
        print("✅ Cleared all annotations (image files preserved)")

    def export_summary(self, output_file: Optional[str] = None) -> str:
        """
        Export a summary of annotations to a text file.

        Args:
            output_file: Path to output file (default: 'annotation_summary.txt' in output_dir)

        Returns:
            Path to the summary file
        """
        if output_file is None:
            output_file = os.path.join(self.output_dir, "annotation_summary.txt")

        stats = self.get_stats()

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write("ANNOTATION SUMMARY\n")
            f.write("=" * 60 + "\n\n")

            for key, value in stats.items():
                f.write(f"{key}: {value}\n")

            f.write("\n" + "=" * 60 + "\n")
            f.write("IMAGES:\n")
            f.write("=" * 60 + "\n")

            for img in self.annotations['images']:
                f.write(f"ID: {img['id']}, File: {img['file_name']}, "
                       f"Size: {img['width']}x{img['height']}\n")

        print(f"✅ Summary exported to: {output_file}")
        return output_file

# MEDIAPIPE

In [ ]:
# @title
# # @title
# !pip install mediapipe albumentations pycocotools
# import mediapipe as mp
# from typing import Tuple, Optional


# class BodyLandmarkDetector:
#     """
#     Class để phát hiện và đánh dấu các điểm lưng và hông trên ảnh RGB
#     sử dụng MediaPipe Pose
#     """

#     def __init__(self,
#                  min_detection_confidence: float = 0.5,
#                  min_tracking_confidence: float = 0.5):
#         """
#         Khởi tạo detector

#         Args:
#             min_detection_confidence: Ngưỡng tin cậy tối thiểu cho việc phát hiện
#             min_tracking_confidence: Ngưỡng tin cậy tối thiểu cho việc tracking
#         """
#         self.mp_pose = mp.solutions.pose
#         self.mp_drawing = mp.solutions.drawing_utils
#         self.mp_drawing_styles = mp.solutions.drawing_styles

#         self.pose = self.mp_pose.Pose(
#             min_detection_confidence=min_detection_confidence,
#             min_tracking_confidence=min_tracking_confidence,
#             model_complexity=1
#         )

#         # Định nghĩa các điểm landmark cho lưng và hông
#         self.back_hip_landmarks = {
#             'LEFT_SHOULDER': self.mp_pose.PoseLandmark.LEFT_SHOULDER,
#             'RIGHT_SHOULDER': self.mp_pose.PoseLandmark.RIGHT_SHOULDER,
#             'LEFT_HIP': self.mp_pose.PoseLandmark.LEFT_HIP,
#             'RIGHT_HIP': self.mp_pose.PoseLandmark.RIGHT_HIP,
#         }

#     def process_image(self, image: np.ndarray) -> np.ndarray:
#         """
#         Xử lý ảnh và đánh dấu các điểm lưng, hông

#         Args:
#             image: Ảnh RGB đầu vào (numpy array)

#         Returns:
#             Ảnh đã được đánh dấu với tọa độ các keypoints
#         """
#         # Copy ảnh để vẽ
#         annotated_image = image.copy()

#         # Chuyển BGR sang RGB nếu cần (OpenCV đọc ảnh dạng BGR)
#         image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if len(image.shape) == 3 else image

#         # Xử lý ảnh với MediaPipe
#         results = self.pose.process(image_rgb)

#         if results.pose_landmarks:
#             # Vẽ tất cả các điểm pose
#             self.mp_drawing.draw_landmarks(
#                 annotated_image,
#                 results.pose_landmarks,
#                 self.mp_pose.POSE_CONNECTIONS,
#                 landmark_drawing_spec=self.mp_drawing_styles.get_default_pose_landmarks_style()
#             )

#             # Lấy tọa độ các điểm lưng và hông
#             landmarks_dict = self._extract_landmarks(results.pose_landmarks, image.shape)

#             # Vẽ highlight cho các điểm lưng và hông
#             self._highlight_back_hip_points(annotated_image, landmarks_dict)

#             # Vẽ vùng lưng và hông
#             self._draw_back_hip_region(annotated_image, landmarks_dict)

#             # Vẽ tọa độ các điểm lên ảnh
#             self._draw_coordinates_on_image(annotated_image, landmarks_dict)
#         else:
#             # Thông báo không phát hiện được
#             self._draw_no_detection_message(annotated_image)

#         return annotated_image

#     def get_keypoints(self, image: np.ndarray) -> Optional[dict]:
#         """
#         Lấy tọa độ các keypoints của lưng và hông

#         Args:
#             image: Ảnh RGB đầu vào (numpy array)

#         Returns:
#             Dictionary chứa tọa độ các điểm (hoặc None nếu không phát hiện được)
#         """
#         # Chuyển BGR sang RGB nếu cần
#         image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) if len(image.shape) == 3 else image

#         # Xử lý ảnh với MediaPipe
#         results = self.pose.process(image_rgb)

#         if results.pose_landmarks:
#             return self._extract_landmarks(results.pose_landmarks, image.shape)

#         return None

#     def _extract_landmarks(self, pose_landmarks, image_shape: Tuple) -> dict:
#         """
#         Trích xuất tọa độ các điểm landmark

#         Args:
#             pose_landmarks: Kết quả pose landmarks từ MediaPipe
#             image_shape: Kích thước ảnh (height, width, channels)

#         Returns:
#             Dictionary chứa tọa độ các điểm
#         """
#         h, w = image_shape[:2]
#         landmarks = {}

#         for name, landmark_id in self.back_hip_landmarks.items():
#             landmark = pose_landmarks.landmark[landmark_id]
#             x = int(landmark.x * w)
#             y = int(landmark.y * h)
#             visibility = landmark.visibility
#             landmarks[name] = {
#                 'x': x,
#                 'y': y,
#                 'visibility': visibility
#             }

#         return landmarks

#     def _highlight_back_hip_points(self, image: np.ndarray, landmarks: dict):
#         """
#         Highlight các điểm lưng và hông bằng vòng tròn màu sắc nổi bật

#         Args:
#             image: Ảnh để vẽ
#             landmarks: Dictionary chứa tọa độ các điểm
#         """
#         colors = {
#             'LEFT_SHOULDER': (255, 0, 0),    # Xanh dương
#             'RIGHT_SHOULDER': (255, 0, 0),   # Xanh dương
#             'LEFT_HIP': (0, 255, 0),         # Xanh lá
#             'RIGHT_HIP': (0, 255, 0),        # Xanh lá
#         }

#         for name, coords in landmarks.items():
#             if coords['visibility'] > 0.5:  # Chỉ vẽ nếu điểm rõ ràng
#                 cv2.circle(image, (coords['x'], coords['y']), 10, colors[name], -1)
#                 cv2.circle(image, (coords['x'], coords['y']), 12, (255, 255, 255), 2)

#                 # Vẽ label
#                 cv2.putText(image, name.replace('_', ' '),
#                            (coords['x'] + 15, coords['y']),
#                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[name], 2)

#     def _draw_back_hip_region(self, image: np.ndarray, landmarks: dict):
#         """
#         Vẽ vùng lưng và hông bằng các đường kết nối

#         Args:
#             image: Ảnh để vẽ
#             landmarks: Dictionary chứa tọa độ các điểm
#         """
#         # Vẽ đường kết nối giữa vai (phần trên lưng)
#         if (landmarks['LEFT_SHOULDER']['visibility'] > 0.5 and
#             landmarks['RIGHT_SHOULDER']['visibility'] > 0.5):
#             cv2.line(image,
#                     (landmarks['LEFT_SHOULDER']['x'], landmarks['LEFT_SHOULDER']['y']),
#                     (landmarks['RIGHT_SHOULDER']['x'], landmarks['RIGHT_SHOULDER']['y']),
#                     (0, 255, 255), 3)

#         # Vẽ đường kết nối giữa hông
#         if (landmarks['LEFT_HIP']['visibility'] > 0.5 and
#             landmarks['RIGHT_HIP']['visibility'] > 0.5):
#             cv2.line(image,
#                     (landmarks['LEFT_HIP']['x'], landmarks['LEFT_HIP']['y']),
#                     (landmarks['RIGHT_HIP']['x'], landmarks['RIGHT_HIP']['y']),
#                     (0, 255, 255), 3)

#         # Vẽ đường kết nối từ vai xuống hông (cột sống)
#         if (landmarks['LEFT_SHOULDER']['visibility'] > 0.5 and
#             landmarks['LEFT_HIP']['visibility'] > 0.5):
#             cv2.line(image,
#                     (landmarks['LEFT_SHOULDER']['x'], landmarks['LEFT_SHOULDER']['y']),
#                     (landmarks['LEFT_HIP']['x'], landmarks['LEFT_HIP']['y']),
#                     (255, 0, 255), 2)

#         if (landmarks['RIGHT_SHOULDER']['visibility'] > 0.5 and
#             landmarks['RIGHT_HIP']['visibility'] > 0.5):
#             cv2.line(image,
#                     (landmarks['RIGHT_SHOULDER']['x'], landmarks['RIGHT_SHOULDER']['y']),
#                     (landmarks['RIGHT_HIP']['x'], landmarks['RIGHT_HIP']['y']),
#                     (255, 0, 255), 2)

#     def _draw_coordinates_on_image(self, image: np.ndarray, landmarks: dict):
#         """
#         Vẽ tọa độ các điểm lên ảnh

#         Args:
#             image: Ảnh để vẽ
#             landmarks: Dictionary chứa tọa độ các điểm
#         """
#         # Tạo box nền cho text
#         cv2.rectangle(image, (10, 10), (400, 180), (0, 0, 0), -1)
#         cv2.rectangle(image, (10, 10), (400, 180), (255, 255, 255), 2)

#         # Tiêu đề
#         cv2.putText(image, "TOA DO CAC DIEM LUNG VA HONG",
#                    (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

#         # Vẽ tọa độ từng điểm
#         y_offset = 55
#         for name, coords in landmarks.items():
#             text = f"{name}: ({coords['x']}, {coords['y']})"
#             cv2.putText(image, text,
#                        (15, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
#             y_offset += 25

#     def _draw_no_detection_message(self, image: np.ndarray):
#         """
#         Vẽ thông báo không phát hiện được người

#         Args:
#             image: Ảnh để vẽ
#         """
#         cv2.rectangle(image, (10, 10), (350, 50), (0, 0, 255), -1)
#         cv2.putText(image, "Khong phat hien duoc nguoi",
#                    (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

#     def __del__(self):
#         """Giải phóng tài nguyên"""
#         self.pose.close()



# MODAL SEGMENTATION BACK

In [ ]:
# @title
# ============================================
# U-NET++ MODEL (Nested U-Net)
# ============================================
class BackSegmentationModel:
    """
    Model phát hiện vùng lưng sử dụng UNet++ với backbone pre-trained
    """
    def __init__(self, backbone='resnet34', encoder_weights='imagenet', device='cuda'):
        """
        Args:
            backbone: Backbone network (resnet34, resnet50, efficientnet-b0, etc.)
            encoder_weights: Pre-trained weights ('imagenet' hoặc None)
            device: 'cuda' hoặc 'cpu'
        """
        self.device = device if torch.cuda.is_available() else 'cpu'

        # Khởi tạo model UNet++
        self.model = smp.UnetPlusPlus(
            encoder_name=backbone,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,  # Binary segmentation
            activation=None  # Sử dụng sigmoid trong loss
        )
        self.model.to(self.device)

        # Loss function và optimizer
        self.criterion = smp.losses.DiceLoss(mode='binary')
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-4)

    def get_training_augmentation(self):
        """Augmentation cho training"""
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(scale_limit=0.1, rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.Resize(512, 512),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def get_validation_augmentation(self):
        """Augmentation cho validation"""
        return A.Compose([
            A.Resize(512, 512),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def train_epoch(self, train_loader):
        """Train một epoch"""
        self.model.train()
        total_loss = 0

        for images, masks in train_loader:
            images = images.to(self.device)
            masks = masks.to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, masks)

            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()

        return total_loss / len(train_loader)

    def validate(self, val_loader):
        """Validate model"""
        self.model.eval()
        total_loss = 0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(self.device)
                masks = masks.to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                total_loss += loss.item()

        return total_loss / len(val_loader)

    def predict(self, image_tensor=None, image_rgb=None, threshold=0.95):
        """
        Predict mask từ:
          - image_tensor: Tensor dạng (B,3,H,W) hoặc (3,H,W)
          - image_path: đường dẫn tới ảnh (OpenCV đọc ảnh)

        Trả về:
          - mask: Tensor (B,1,H,W)
        """

        self.model.eval()

        if image_tensor is not None:
            if not isinstance(image_tensor, torch.Tensor):
                raise TypeError("image_tensor must be a torch.Tensor")

            # Nếu là (3,H,W) → thêm batch dim
            if image_tensor.dim() == 3:
                image_tensor = image_tensor.unsqueeze(0)

            # Đảm bảo đúng dạng (B,3,H,W)
            assert image_tensor.dim() == 4, "image_tensor must be (B,3,H,W)"

            image_tensor = image_tensor.float().to(self.device)

            with torch.no_grad():
                output = self.model(image_tensor)
                output = torch.sigmoid(output)
                mask = (output > threshold).float()

            return mask    # Tensor (B,1,H,W)

        elif image_rgb is not None:

            image_rgb = self.pad_to_32(image_rgb)

            H, W = image_rgb.shape[:2]

            # Chuyển sang tensor (3,H,W)
            image_t = torch.from_numpy(image_rgb).permute(2,0,1).float() / 255.0
            image_t = image_t.unsqueeze(0).to(self.device)

            with torch.no_grad():
                output = self.model(image_t)
                output = torch.sigmoid(output)[0,0].cpu().numpy()

            # Resize về đúng kích thước ảnh gốc
            mask = cv2.resize(output, (W, H))

            # Binary mask
            mask_binary = (mask > threshold).astype("uint8")

            return mask_binary   # numpy(H,W)

        else:
            raise ValueError("You must provide either image_tensor or image_path")

    def pad_to_32(self, image):
        h, w = image.shape[:2]

        new_h = math.ceil(h / 32) * 32
        new_w = math.ceil(w / 32) * 32

        pad_bottom = new_h - h
        pad_right  = new_w - w

        padded = cv2.copyMakeBorder(
            image,
            top=0, bottom=pad_bottom,
            left=0, right=pad_right,
            borderType=cv2.BORDER_CONSTANT,
            value=(0,0,0)
        )
        return padded

    def save_model(self, path):
        """Lưu model"""
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
        }, path)
        print(f"Model đã được lưu tại: {path}")

    def load_model(self, path):
        """Load model"""
        checkpoint = torch.load(path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        print(f"Model đã được load từ: {path}")


# TRAIN

In [ ]:
# # @title
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader
# from tqdm import tqdm
# import os
# import matplotlib.pyplot as plt
# import numpy as np
# from typing import List, Dict, Optional, Tuple
# from scipy.ndimage import distance_transform_edt
# import gc


# # ============================================================================
# # AGGRESSIVE MEMORY MANAGEMENT
# # ============================================================================
# def clear_memory(verbose=False):
#     """Xóa bộ nhớ CUDA triệt để"""
#     if verbose:
#         before = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

#     # Xóa cache Python
#     gc.collect()

#     # Xóa cache CUDA
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.synchronize()
#         torch.cuda.ipc_collect()  # Thêm IPC collect

#     if verbose and torch.cuda.is_available():
#         after = torch.cuda.memory_allocated() / 1024**3
#         print(f"🧹 Memory: {before:.2f}GB → {after:.2f}GB (freed {before-after:.2f}GB)")


# def print_memory_summary(device='cuda'):
#     """In chi tiết bộ nhớ GPU"""
#     if not torch.cuda.is_available():
#         return

#     allocated = torch.cuda.memory_allocated(device) / 1024**3
#     reserved = torch.cuda.memory_reserved(device) / 1024**3
#     max_allocated = torch.cuda.max_memory_allocated(device) / 1024**3
#     total = torch.cuda.get_device_properties(device).total_memory / 1024**3

#     print(f"\n{'='*60}")
#     print(f"GPU Memory (Device: {device})")
#     print(f"{'='*60}")
#     print(f"Total:     {total:.2f} GB")
#     print(f"Allocated: {allocated:.2f} GB ({allocated/total*100:.1f}%)")
#     print(f"Reserved:  {reserved:.2f} GB ({reserved/total*100:.1f}%)")
#     print(f"Peak:      {max_allocated:.2f} GB ({max_allocated/total*100:.1f}%)")
#     print(f"Free:      {total - reserved:.2f} GB ({(total-reserved)/total*100:.1f}%)")
#     print(f"{'='*60}\n")


# # ============================================================================
# # MEMORY-EFFICIENT DATA LOADING
# # ============================================================================
# class MemoryEfficientDataLoader:
#     """DataLoader tối ưu bộ nhớ với prefetching giới hạn"""

#     def __init__(self, dataset, batch_size, shuffle=True, num_workers=2,
#                  pin_memory=False, drop_last=True):
#         self.loader = DataLoader(
#             dataset,
#             batch_size=batch_size,
#             shuffle=shuffle,
#             num_workers=num_workers,  # Giảm workers
#             pin_memory=pin_memory,    # Tắt pin_memory nếu RAM thấp
#             drop_last=drop_last,
#             prefetch_factor=1 if num_workers > 0 else None,  # Giảm prefetch
#             persistent_workers=False  # Không giữ workers
#         )

#     def __iter__(self):
#         return iter(self.loader)

#     def __len__(self):
#         return len(self.loader)


# # ============================================================================
# # DISTANCE FIELD LOSS - MEMORY OPTIMIZED
# # ============================================================================
# class DistanceFieldLoss(nn.Module):
#     """Loss function tối ưu bộ nhớ cho distance field regression"""

#     def __init__(self,
#                  use_mse=True,
#                  use_l1=True,
#                  use_gradient=True,
#                  use_peak=True,
#                  mse_weight=1.0,
#                  l1_weight=0.5,
#                  gradient_weight=0.3,
#                  peak_weight=0.2):
#         super().__init__()
#         self.use_mse = use_mse
#         self.use_l1 = use_l1
#         self.use_gradient = use_gradient
#         self.use_peak = use_peak

#         self.mse_weight = mse_weight
#         self.l1_weight = l1_weight
#         self.gradient_weight = gradient_weight
#         self.peak_weight = peak_weight

#     def forward(self, pred, target, deep_sup_preds=None):
#         """Tính loss với giải phóng bộ nhớ ngay lập tức"""
#         loss = torch.tensor(0.0, device=pred.device, requires_grad=True)
#         loss_dict = {}

#         # Resize nếu cần
#         if pred.shape[-2:] != target.shape[-2:]:
#             pred = F.interpolate(pred, size=target.shape[-2:],
#                                mode='bilinear', align_corners=False)

#         # MSE Loss
#         if self.use_mse:
#             mse_loss = F.mse_loss(pred, target)
#             loss = loss + self.mse_weight * mse_loss
#             loss_dict['mse'] = mse_loss.item()
#             del mse_loss

#         # L1 Loss
#         if self.use_l1:
#             l1_loss = F.l1_loss(pred, target)
#             loss = loss + self.l1_weight * l1_loss
#             loss_dict['l1'] = l1_loss.item()
#             del l1_loss

#         # Gradient Loss
#         if self.use_gradient:
#             pred_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]
#             target_dx = target[:, :, :, 1:] - target[:, :, :, :-1]
#             pred_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]
#             target_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

#             grad_loss = F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)
#             loss = loss + self.gradient_weight * grad_loss
#             loss_dict['gradient'] = grad_loss.item()
#             del pred_dx, target_dx, pred_dy, target_dy, grad_loss

#         # Peak Loss
#         if self.use_peak:
#             peak_mask = (target > 0.7).float()
#             if peak_mask.sum() > 0:
#                 peak_error = ((pred - target) ** 2) * peak_mask
#                 peak_loss = peak_error.sum() / (peak_mask.sum() + 1e-7)
#                 loss = loss + self.peak_weight * peak_loss
#                 loss_dict['peak'] = peak_loss.item()
#                 del peak_error, peak_loss
#             else:
#                 loss_dict['peak'] = 0.0
#             del peak_mask

#         # Deep supervision - giản lược
#         if deep_sup_preds is not None and len(deep_sup_preds) > 0:
#             ds_loss_total = torch.tensor(0.0, device=pred.device)
#             for i, ds_pred in enumerate(deep_sup_preds):
#                 if ds_pred.shape[-2:] != target.shape[-2:]:
#                     ds_pred = F.interpolate(ds_pred, size=target.shape[-2:],
#                                           mode='bilinear', align_corners=False)
#                 ds_loss = F.mse_loss(ds_pred, target)
#                 ds_loss_total = ds_loss_total + ds_loss
#                 loss_dict[f'deep_sup_{i}'] = ds_loss.item()
#                 del ds_pred, ds_loss

#             loss = loss + 0.3 * ds_loss_total
#             del ds_loss_total

#         loss_dict['total'] = loss.item()
#         return loss, loss_dict


# # ============================================================================
# # DISTANCE FIELD METRICS
# # ============================================================================
# class DistanceFieldMetrics:
#     """Metrics cho distance field predictions"""

#     @staticmethod
#     def peak_precision(pred, target, threshold=0.7):
#         """Đo độ chính xác của peaks"""
#         pred_peaks = (pred > threshold).float()
#         target_peaks = (target > threshold).float()

#         intersection = (pred_peaks * target_peaks).sum()
#         pred_sum = pred_peaks.sum()

#         precision = intersection / pred_sum if pred_sum > 0 else torch.tensor(0.0)

#         # Giải phóng bộ nhớ
#         del pred_peaks, target_peaks, intersection, pred_sum

#         return precision.item()

#     @staticmethod
#     def curve_continuity(pred, threshold=0.5):
#         """Đo độ liên tục của curve"""
#         pred_binary = (pred > threshold).float()
#         kernel = torch.ones(1, 1, 3, 3, device=pred.device)
#         neighbors = F.conv2d(pred_binary, kernel, padding=1)

#         continuous_pixels = (neighbors >= 2).float().sum()
#         total_pixels = pred_binary.sum() + 1e-7
#         continuity = (continuous_pixels / total_pixels).item()

#         # Giải phóng
#         del pred_binary, kernel, neighbors, continuous_pixels, total_pixels

#         return continuity

#     @staticmethod
#     def mean_distance_error(pred, target):
#         """Mean absolute error"""
#         mae = torch.abs(pred - target).mean().item()
#         return mae


# # ============================================================================
# # MULTI-SCALE INFERENCE - MEMORY OPTIMIZED
# # ============================================================================
# @torch.no_grad()
# def multi_scale_inference(model, images, scales=[1.0], flip=False):
#     """Multi-scale inference tối ưu bộ nhớ"""
#     model.eval()
#     _, _, orig_h, orig_w = images.shape

#     # Khởi tạo với scale đầu tiên
#     scaled_imgs = F.interpolate(images, scale_factor=scales[0],
#                                mode='bilinear', align_corners=False)

#     with torch.cuda.amp.autocast():
#         final_preds, _ = model(scaled_imgs)

#     final_preds = F.interpolate(final_preds, size=(orig_h, orig_w),
#                                mode='bilinear', align_corners=False)
#     count = 1

#     del scaled_imgs

#     # Các scales còn lại
#     for scale in scales[1:]:
#         scaled_imgs = F.interpolate(images, scale_factor=scale,
#                                    mode='bilinear', align_corners=False)

#         with torch.cuda.amp.autocast():
#             pred, _ = model(scaled_imgs)

#         pred = F.interpolate(pred, size=(orig_h, orig_w),
#                            mode='bilinear', align_corners=False)
#         final_preds = final_preds + pred
#         count += 1

#         del scaled_imgs, pred

#         # Flip augmentation
#         if flip:
#             scaled_imgs = F.interpolate(images, scale_factor=scale,
#                                        mode='bilinear', align_corners=False)
#             flipped_imgs = torch.flip(scaled_imgs, dims=[3])

#             with torch.cuda.amp.autocast():
#                 pred_flip, _ = model(flipped_imgs)

#             pred_flip = torch.flip(pred_flip, dims=[3])
#             pred_flip = F.interpolate(pred_flip, size=(orig_h, orig_w),
#                                     mode='bilinear', align_corners=False)
#             final_preds = final_preds + pred_flip
#             count += 1

#             del scaled_imgs, flipped_imgs, pred_flip

#     final_preds = final_preds / count
#     return final_preds


# # ============================================================================
# # EMA - MEMORY OPTIMIZED
# # ============================================================================
# class EMA:
#     """Exponential Moving Average tối ưu bộ nhớ"""

#     def __init__(self, model, decay=0.999):
#         self.model = model
#         self.decay = decay
#         self.shadow = {}
#         self.backup = {}

#         # Chỉ lưu shadow cho parameters cần gradient
#         for name, param in model.named_parameters():
#             if param.requires_grad:
#                 self.shadow[name] = param.data.clone().detach()

#     @torch.no_grad()
#     def update(self):
#         """Update EMA weights"""
#         for name, param in self.model.named_parameters():
#             if param.requires_grad and name in self.shadow:
#                 self.shadow[name].mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

#     def apply_shadow(self):
#         """Áp dụng EMA weights"""
#         for name, param in self.model.named_parameters():
#             if param.requires_grad and name in self.shadow:
#                 self.backup[name] = param.data.clone()
#                 param.data.copy_(self.shadow[name])

#     def restore(self):
#         """Khôi phục weights gốc"""
#         for name, param in self.model.named_parameters():
#             if param.requires_grad and name in self.backup:
#                 param.data.copy_(self.backup[name])
#         self.backup.clear()


# # ============================================================================
# # MEMORY-OPTIMIZED TRAINING FUNCTION
# # ============================================================================
# def train_spine_curve_model(
#     model,
#     train_dataset,
#     val_dataset,
#     device='cuda',
#     epochs=100,
#     batch_size=4,  # Giảm mặc định
#     lr=1e-3,
#     save_path='best_spine_curve_model.pth',
#     plot_save_path='training_metrics.png',
#     model_segmentation_back=None,
#     resume_from=None,
#     # Multi-scale config
#     use_multiscale_val=False,  # Tắt mặc định
#     multiscale_scales=[1.0],   # Chỉ dùng 1 scale
#     multiscale_flip=False,     # Tắt flip
#     # Advanced techniques
#     use_ema=False,             # Tắt EMA mặc định
#     use_mixup=False,
#     mixup_alpha=0.2,
#     # Loss configuration
#     use_gradient_loss=True,
#     use_peak_loss=True,
#     calculate_curve_metrics=False,  # Tắt metrics phức tạp
#     # Memory management - AGGRESSIVE
#     gradient_accumulation_steps=4,  # Tăng accumulation
#     clear_cache_every_n_steps=10,   # Clear thường xuyên hơn
#     clear_cache_after_backward=True,  # Clear sau mỗi backward
#     clear_cache_after_forward=False,  # Clear sau forward nếu cần
# ):
#     """
#     Training function với memory optimization cực mạnh

#     Các tối ưu chính:
#     1. Aggressive cache clearing sau mỗi epoch và mỗi N steps
#     2. Gradient accumulation cao hơn để dùng batch size nhỏ
#     3. Giới hạn workers và prefetch
#     4. Xóa tensors ngay sau khi dùng
#     5. Giảm thiểu deep supervision
#     6. Tắt các features tốn memory (EMA, multiscale) mặc định
#     """

#     print(f"\n{'='*80}")
#     print("🚀 MEMORY-OPTIMIZED TRAINING FOR SPINE CURVE DETECTION")
#     print(f"{'='*80}")

#     # Initial cleanup
#     clear_memory(verbose=True)

#     model = model.to(device)

#     # Memory efficient settings
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True

#     if torch.cuda.is_available():
#         print_memory_summary(device)

#     # Loss function
#     criterion = DistanceFieldLoss(
#         use_mse=True,
#         use_l1=True,
#         use_gradient=use_gradient_loss,
#         use_peak=use_peak_loss,
#         mse_weight=1.0,
#         l1_weight=0.5,
#         gradient_weight=0.3,
#         peak_weight=0.2
#     )

#     # Optimizer
#     optimizer = torch.optim.AdamW(
#         model.parameters(),
#         lr=lr,
#         weight_decay=1e-4,
#         betas=(0.9, 0.999)
#     )

#     # Scheduler
#     scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
#         optimizer, T_0=15, T_mult=2, eta_min=1e-6
#     )

#     # EMA (optional)
#     ema = EMA(model, decay=0.999) if use_ema else None

#     # Metrics
#     metrics_calc = DistanceFieldMetrics() if calculate_curve_metrics else None

#     # History
#     history = {
#         'train_loss': [], 'val_loss': [],
#         'train_mse': [], 'val_mse': [],
#         'lr': []
#     }

#     if use_gradient_loss:
#         history['train_gradient'] = []
#         history['val_gradient'] = []
#     if use_peak_loss:
#         history['train_peak'] = []
#         history['val_peak'] = []
#     if calculate_curve_metrics:
#         history['val_precision'] = []
#         history['val_continuity'] = []

#     # Training state
#     start_epoch = 0
#     best_val_loss = float('inf')
#     patience = 0
#     max_patience = 20

#     # Resume from checkpoint
#     if resume_from and os.path.exists(resume_from):
#         print(f"🔄 Resuming from: {resume_from}")
#         checkpoint = torch.load(resume_from, map_location=device)
#         model.load_state_dict(checkpoint['model_state_dict'])
#         optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#         scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#         start_epoch = checkpoint['epoch'] + 1
#         best_val_loss = checkpoint['best_val_loss']
#         if 'history' in checkpoint:
#             history = checkpoint['history']
#         print(f"✔ Resumed from epoch {start_epoch}")
#         clear_memory(verbose=True)

#     # Data loaders - memory efficient
#     print("\n📦 Creating memory-efficient data loaders...")
#     train_loader = MemoryEfficientDataLoader(
#         train_dataset, batch_size=batch_size, shuffle=True,
#         num_workers=2, pin_memory=False, drop_last=True
#     )
#     val_loader = MemoryEfficientDataLoader(
#         val_dataset, batch_size=batch_size, shuffle=False,
#         num_workers=2, pin_memory=False, drop_last=False
#     )

#     print(f"✔ Train batches: {len(train_loader)}")
#     print(f"✔ Val batches: {len(val_loader)}")
#     print(f"✔ Effective batch size: {batch_size * gradient_accumulation_steps}")
#     print(f"✔ Cache clear frequency: Every {clear_cache_every_n_steps} steps")

#     # Mixed precision scaler
#     scaler = torch.cuda.amp.GradScaler()

#     # ========================================================================
#     # TRAINING LOOP
#     # ========================================================================
#     for epoch in range(start_epoch, epochs):
#         print(f"\n{'='*80}")
#         print(f"Epoch {epoch+1}/{epochs}")
#         print(f"{'='*80}")

#         # ============== TRAINING ==============
#         model.train()
#         train_loss = 0.0
#         train_metrics = {'mse': 0, 'l1': 0, 'gradient': 0, 'peak': 0}

#         optimizer.zero_grad()

#         pbar = tqdm(train_loader, desc="🔧 Training")

#         for step, (images, distance_fields) in enumerate(pbar):
#             images = images.to(device, non_blocking=True)
#             distance_fields = distance_fields.to(device, non_blocking=True)
#             distance_fields = torch.clamp(distance_fields, 0.0, 1.0)

#             # Background masking (optional)
#             if model_segmentation_back is not None:
#                 with torch.no_grad():
#                     mask = model_segmentation_back.predict(image_tensor=images)
#                     if mask.dim() == 3:
#                         mask = mask.unsqueeze(1)
#                     if mask.shape[-2:] != images.shape[-2:]:
#                         mask = F.interpolate(mask, size=images.shape[-2:],
#                                            mode="bilinear", align_corners=False)
#                     images = images * mask
#                     del mask

#             # Forward pass
#             with torch.cuda.amp.autocast():
#                 pred, deep_sup = model(images)

#                 # Giới hạn deep supervision
#                 if deep_sup and len(deep_sup) > 2:
#                     deep_sup = deep_sup[:2]

#                 loss, loss_dict = criterion(pred, distance_fields, deep_sup_preds=deep_sup)
#                 loss = loss / gradient_accumulation_steps

#             # Clear after forward if needed
#             if clear_cache_after_forward:
#                 del pred, deep_sup
#                 clear_memory()

#             # Backward pass
#             scaler.scale(loss).backward()

#             # Clear after backward
#             if clear_cache_after_backward:
#                 clear_memory()

#             # Update weights
#             if (step + 1) % gradient_accumulation_steps == 0:
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
#                 optimizer.zero_grad()

#                 if ema is not None:
#                     ema.update()

#             # Accumulate metrics
#             train_loss += loss.item() * gradient_accumulation_steps
#             for k in ['mse', 'l1', 'gradient', 'peak']:
#                 train_metrics[k] += loss_dict.get(k, 0)

#             pbar.set_postfix({'loss': f"{loss.item() * gradient_accumulation_steps:.4f}"})

#             # Periodic cache clearing
#             if (step + 1) % clear_cache_every_n_steps == 0:
#                 clear_memory()

#         # Average training metrics
#         n = len(train_loader)
#         train_loss /= n
#         for k in train_metrics:
#             train_metrics[k] /= n

#         # AGGRESSIVE cleanup after training
#         del images, distance_fields, loss, loss_dict
#         clear_memory(verbose=True)

#         # ============== VALIDATION ==============
#         model.eval()
#         val_loss = 0.0
#         val_metrics = {'mse': 0, 'l1': 0, 'gradient': 0, 'peak': 0}
#         val_precision = 0.0
#         val_continuity = 0.0

#         with torch.no_grad():
#             for batch_idx, (images, distance_fields) in enumerate(tqdm(val_loader, desc="🔍 Validation")):
#                 images = images.to(device, non_blocking=True)
#                 distance_fields = distance_fields.to(device, non_blocking=True)
#                 distance_fields = torch.clamp(distance_fields, 0.0, 1.0)

#                 # Forward
#                 with torch.cuda.amp.autocast():
#                     pred, _ = model(images)
#                     loss, loss_dict = criterion(pred, distance_fields, deep_sup_preds=None)

#                 val_loss += loss.item()
#                 for k in ['mse', 'l1', 'gradient', 'peak']:
#                     val_metrics[k] += loss_dict.get(k, 0)

#                 # Metrics
#                 if calculate_curve_metrics:
#                     val_precision += metrics_calc.peak_precision(pred, distance_fields)
#                     val_continuity += metrics_calc.curve_continuity(pred)

#                 # Cleanup
#                 del pred, loss, loss_dict

#                 # Clear every few batches
#                 if (batch_idx + 1) % 5 == 0:
#                     clear_memory()

#         # Average validation metrics
#         m = len(val_loader)
#         val_loss /= m
#         for k in val_metrics:
#             val_metrics[k] /= m

#         if calculate_curve_metrics:
#             val_precision /= m
#             val_continuity /= m

#         # AGGRESSIVE cleanup after validation
#         del images, distance_fields
#         clear_memory(verbose=True)

#         # Update scheduler
#         lr_current = optimizer.param_groups[0]['lr']
#         scheduler.step()

#         # Save history
#         history['train_loss'].append(train_loss)
#         history['val_loss'].append(val_loss)
#         history['train_mse'].append(train_metrics['mse'])
#         history['val_mse'].append(val_metrics['mse'])
#         history['lr'].append(lr_current)

#         if use_gradient_loss:
#             history['train_gradient'].append(train_metrics['gradient'])
#             history['val_gradient'].append(val_metrics['gradient'])
#         if use_peak_loss:
#             history['train_peak'].append(train_metrics['peak'])
#             history['val_peak'].append(val_metrics['peak'])
#         if calculate_curve_metrics:
#             history['val_precision'].append(val_precision)
#             history['val_continuity'].append(val_continuity)

#         # ============== LOGGING ==============
#         print(f"\n📊 Results:")
#         print(f"Train | Loss: {train_loss:.4f} | MSE: {train_metrics['mse']:.4f}")
#         print(f"Val   | Loss: {val_loss:.4f} | MSE: {val_metrics['mse']:.4f}")

#         if calculate_curve_metrics:
#             print(f"      | Precision: {val_precision:.4f} | Continuity: {val_continuity:.4f}")

#         print(f"      | LR: {lr_current:.6f}")

#         if torch.cuda.is_available():
#             current = torch.cuda.memory_allocated(device) / 1024**3
#             peak = torch.cuda.max_memory_allocated(device) / 1024**3
#             print(f"      | GPU: Current={current:.2f}GB | Peak={peak:.2f}GB")

#         # ============== CHECKPOINTING ==============
#         if val_loss < best_val_loss:
#             best_val_loss = val_loss
#             patience = 0

#             checkpoint = {
#                 'epoch': epoch,
#                 'model_state_dict': model.state_dict(),
#                 'optimizer_state_dict': optimizer.state_dict(),
#                 'scheduler_state_dict': scheduler.state_dict(),
#                 'val_loss': val_loss,
#                 'best_val_loss': best_val_loss,
#                 'history': history,
#                 'config': {
#                     'batch_size': batch_size,
#                     'gradient_accumulation_steps': gradient_accumulation_steps,
#                 }
#             }

#             torch.save(checkpoint, save_path)
#             print("✅ Best model saved!")
#         else:
#             patience += 1
#             print(f"⏳ Patience: {patience}/{max_patience}")

#         # Early stopping
#         if patience >= max_patience:
#             print("\n⛔ Early stopping!")
#             break

#         # Plot metrics
#         # try:
#         #     plot_training_metrics(history, plot_save_path)
#         # except:
#         #     pass

#         # AGGRESSIVE cleanup after epoch
#         torch.cuda.reset_peak_memory_stats(device)
#         clear_memory(verbose=True)

#         print_memory_summary(device)

#     # Final cleanup
#     print("\n" + "="*80)
#     print("🎉 Training Complete!")
#     print(f"📁 Best model: {save_path}")
#     print(f"📉 Best val loss: {best_val_loss:.4f}")
#     print("="*80)

#     clear_memory(verbose=True)

#     return history


# # ============================================================================
# # PLOTTING
# # ============================================================================

# def plot_training_metrics(history, save_path='training_metrics.png'):
#     """Plot training metrics"""
#     os.makedirs(os.path.dirname(save_path), exist_ok=True)

#     epochs = range(1, len(history['train_loss']) + 1)

#     fig, axes = plt.subplots(2, 2, figsize=(12, 10))

#     axes[0, 0].plot(epochs, history['train_loss'], label='Train', linewidth=2)
#     axes[0, 0].plot(epochs, history['val_loss'], label='Val', linewidth=2)
#     axes[0, 0].set_title('Total Loss')
#     axes[0, 0].legend()
#     axes[0, 0].grid(alpha=0.3)

#     axes[0, 1].plot(epochs, history['train_mse'], label='Train', linewidth=2)
#     axes[0, 1].plot(epochs, history['val_mse'], label='Val', linewidth=2)
#     axes[0, 1].set_title('MSE Loss')
#     axes[0, 1].legend()
#     axes[0, 1].grid(alpha=0.3)

#     # axes[1, 0].plot(epochs, history['lr'], linewidth=2)
#     # axes[1, 0].set_title('Learning Rate')
#     # axes[1, 0].set_yscale('log')
#     # axes[1, 0].grid(alpha=0.3)

#     # if 'val_precision' in history and 'val_continuity' in history:
#     #     axes[1, 1].plot(epochs, history['val_precision'], label='Precision', linewidth=2)
#     #     axes[1, 1].plot(epochs, history['val_continuity'], label='Continuity', linewidth=2)
#     #     axes[1, 1].set_title('Curve Metrics')
#     #     axes[1, 1].legend()
#     #     axes[1, 1].grid(alpha=0.3)
#     # else:
#     #     axes[1, 1].axis('off')

#     plt.tight_layout()
#     plt.savefig(save_path, dpi=300)
#     plt.show()


acc

In [ ]:
# @title
# ============================================================================
# DATA AUGMENTATION FOR DISTANCE FIELD
# ============================================================================
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import random
from torch.utils.data import DataLoader
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Optional, Tuple
from scipy.ndimage import distance_transform_edt
import gc

def clear_memory(verbose=False):
    """Xóa bộ nhớ CUDA triệt để"""
    if verbose:
        before = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0

    # Xóa cache Python
    gc.collect()

    # Xóa cache CUDA
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.ipc_collect()  # Thêm IPC collect

    if verbose and torch.cuda.is_available():
        after = torch.cuda.memory_allocated() / 1024**3
        print(f"🧹 Memory: {before:.2f}GB → {after:.2f}GB (freed {before-after:.2f}GB)")


def print_memory_summary(device='cuda'):
    """In chi tiết bộ nhớ GPU"""
    if not torch.cuda.is_available():
        return

    allocated = torch.cuda.memory_allocated(device) / 1024**3
    reserved = torch.cuda.memory_reserved(device) / 1024**3
    max_allocated = torch.cuda.max_memory_allocated(device) / 1024**3
    total = torch.cuda.get_device_properties(device).total_memory / 1024**3

    print(f"\n{'='*60}")
    print(f"GPU Memory (Device: {device})")
    print(f"{'='*60}")
    print(f"Total:     {total:.2f} GB")
    print(f"Allocated: {allocated:.2f} GB ({allocated/total*100:.1f}%)")
    print(f"Reserved:  {reserved:.2f} GB ({reserved/total*100:.1f}%)")
    print(f"Peak:      {max_allocated:.2f} GB ({max_allocated/total*100:.1f}%)")
    print(f"Free:      {total - reserved:.2f} GB ({(total-reserved)/total*100:.1f}%)")
    print(f"{'='*60}\n")


# ============================================================================
# MEMORY-EFFICIENT DATA LOADING
# ============================================================================
class MemoryEfficientDataLoader:
    """DataLoader tối ưu bộ nhớ với prefetching giới hạn"""

    def __init__(self, dataset, batch_size, shuffle=True, num_workers=2,
                 pin_memory=False, drop_last=True):
        self.loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,  # Giảm workers
            pin_memory=pin_memory,    # Tắt pin_memory nếu RAM thấp
            drop_last=drop_last,
            prefetch_factor=1 if num_workers > 0 else None,  # Giảm prefetch
            persistent_workers=False  # Không giữ workers
        )

    def __iter__(self):
        return iter(self.loader)

    def __len__(self):
        return len(self.loader)


class DistanceFieldAugmentation:
    """Augmentation cho cả image và distance field"""

    def __init__(self,
                 p_hflip=0.5,
                 p_vflip=0.3,
                 p_rotate=0.5,
                 rotation_range=(-15, 15),
                 p_brightness=0.4,
                 brightness_range=(0.8, 1.2),
                 p_contrast=0.4,
                 contrast_range=(0.8, 1.2),
                 p_noise=0.3,
                 noise_std=0.02,
                 p_blur=0.2):
        """
        Args:
            p_hflip: probability of horizontal flip
            p_vflip: probability of vertical flip
            p_rotate: probability of rotation
            rotation_range: tuple (min_angle, max_angle) in degrees
            p_brightness: probability of brightness adjustment
            brightness_range: tuple (min_factor, max_factor)
            p_contrast: probability of contrast adjustment
            contrast_range: tuple (min_factor, max_factor)
            p_noise: probability of adding noise
            noise_std: standard deviation of Gaussian noise
            p_blur: probability of Gaussian blur
        """
        self.p_hflip = p_hflip
        self.p_vflip = p_vflip
        self.p_rotate = p_rotate
        self.rotation_range = rotation_range
        self.p_brightness = p_brightness
        self.brightness_range = brightness_range
        self.p_contrast = p_contrast
        self.contrast_range = contrast_range
        self.p_noise = p_noise
        self.noise_std = noise_std
        self.p_blur = p_blur

    def __call__(self, image, distance_field):
        """
        Apply augmentation to both image and distance field

        Args:
            image: [C, H, W] tensor
            distance_field: [1, H, W] tensor

        Returns:
            augmented_image, augmented_distance_field
        """
        # Horizontal flip
        if random.random() < self.p_hflip:
            image = TF.hflip(image)
            distance_field = TF.hflip(distance_field)

        # Vertical flip
        if random.random() < self.p_vflip:
            image = TF.vflip(image)
            distance_field = TF.vflip(distance_field)

        # Rotation (áp dụng cho cả image và distance field)
        if random.random() < self.p_rotate:
            angle = random.uniform(*self.rotation_range)
            image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
            distance_field = TF.rotate(distance_field, angle, interpolation=TF.InterpolationMode.BILINEAR)

        # Brightness (chỉ áp dụng cho image)
        if random.random() < self.p_brightness:
            factor = random.uniform(*self.brightness_range)
            image = TF.adjust_brightness(image, factor)

        # Contrast (chỉ áp dụng cho image)
        if random.random() < self.p_contrast:
            factor = random.uniform(*self.contrast_range)
            image = TF.adjust_contrast(image, factor)

        # Gaussian noise (chỉ áp dụng cho image)
        if random.random() < self.p_noise:
            noise = torch.randn_like(image) * self.noise_std
            image = image + noise
            image = torch.clamp(image, 0, 1)

        # Gaussian blur (chỉ áp dụng cho image)
        if random.random() < self.p_blur:
            kernel_size = random.choice([3, 5])
            sigma = random.uniform(0.1, 2.0)
            image = TF.gaussian_blur(image, kernel_size, sigma)

        return image, distance_field


# ============================================================================
# DISTANCE FIELD LOSS WITH ACCURACY - MEMORY OPTIMIZED
# ============================================================================
class DistanceFieldLoss(nn.Module):
    """Loss function với tính năng accuracy cho distance field regression"""

    def __init__(self,
                 use_mse=True,
                 use_l1=True,
                 use_gradient=True,
                 use_peak=True,
                 mse_weight=1.0,
                 l1_weight=0.5,
                 gradient_weight=0.3,
                 peak_weight=0.2,
                 accuracy_thresholds=[0.05, 0.1, 0.15]):
        super().__init__()
        self.use_mse = use_mse
        self.use_l1 = use_l1
        self.use_gradient = use_gradient
        self.use_peak = use_peak

        self.mse_weight = mse_weight
        self.l1_weight = l1_weight
        self.gradient_weight = gradient_weight
        self.peak_weight = peak_weight

        self.accuracy_thresholds = accuracy_thresholds

    def calculate_accuracy(self, pred, target):
        """Tính accuracy dựa trên absolute error thresholds"""
        accuracy_dict = {}

        with torch.no_grad():
            abs_error = torch.abs(pred - target)

            for thresh in self.accuracy_thresholds:
                correct = (abs_error < thresh).float()
                accuracy = correct.mean().item()
                accuracy_dict[f'acc_{thresh:.2f}'] = accuracy
                del correct

            mae = abs_error.mean().item()
            accuracy_dict['mae'] = mae

            pred_flat = pred.flatten()
            target_flat = target.flatten()

            if len(pred_flat) > 1:
                correlation = torch.corrcoef(torch.stack([pred_flat, target_flat]))[0, 1]
                accuracy_dict['correlation'] = correlation.item()
            else:
                accuracy_dict['correlation'] = 0.0

            del abs_error, pred_flat, target_flat

        return accuracy_dict

    def forward(self, pred, target, deep_sup_preds=None, compute_accuracy=True):
        """Tính loss và accuracy"""
        loss = torch.tensor(0.0, device=pred.device, requires_grad=True)
        loss_dict = {}
        accuracy_dict = {}

        if pred.shape[-2:] != target.shape[-2:]:
            pred = F.interpolate(pred, size=target.shape[-2:],
                               mode='bilinear', align_corners=False)

        # MSE Loss
        if self.use_mse:
            mse_loss = F.mse_loss(pred, target)
            loss = loss + self.mse_weight * mse_loss
            loss_dict['mse'] = mse_loss.item()
            del mse_loss

        # L1 Loss
        if self.use_l1:
            l1_loss = F.l1_loss(pred, target)
            loss = loss + self.l1_weight * l1_loss
            loss_dict['l1'] = l1_loss.item()
            del l1_loss

        # Gradient Loss
        if self.use_gradient:
            pred_dx = pred[:, :, :, 1:] - pred[:, :, :, :-1]
            target_dx = target[:, :, :, 1:] - target[:, :, :, :-1]
            pred_dy = pred[:, :, 1:, :] - pred[:, :, :-1, :]
            target_dy = target[:, :, 1:, :] - target[:, :, :-1, :]

            grad_loss = F.l1_loss(pred_dx, target_dx) + F.l1_loss(pred_dy, target_dy)
            loss = loss + self.gradient_weight * grad_loss
            loss_dict['gradient'] = grad_loss.item()
            del pred_dx, target_dx, pred_dy, target_dy, grad_loss

        # Peak Loss
        if self.use_peak:
            peak_mask = (target > 0.7).float()
            if peak_mask.sum() > 0:
                peak_error = ((pred - target) ** 2) * peak_mask
                peak_loss = peak_error.sum() / (peak_mask.sum() + 1e-7)
                loss = loss + self.peak_weight * peak_loss
                loss_dict['peak'] = peak_loss.item()
                del peak_error, peak_loss
            else:
                loss_dict['peak'] = 0.0
            del peak_mask

        # Deep supervision
        if deep_sup_preds is not None and len(deep_sup_preds) > 0:
            ds_loss_total = torch.tensor(0.0, device=pred.device)
            for i, ds_pred in enumerate(deep_sup_preds):
                if ds_pred.shape[-2:] != target.shape[-2:]:
                    ds_pred = F.interpolate(ds_pred, size=target.shape[-2:],
                                          mode='bilinear', align_corners=False)
                ds_loss = F.mse_loss(ds_pred, target)
                ds_loss_total = ds_loss_total + ds_loss
                loss_dict[f'deep_sup_{i}'] = ds_loss.item()
                del ds_pred, ds_loss

            loss = loss + 0.3 * ds_loss_total
            del ds_loss_total

        loss_dict['total'] = loss.item()

        if compute_accuracy:
            accuracy_dict = self.calculate_accuracy(pred, target)

        return loss, loss_dict, accuracy_dict


# ============================================================================
# MULTISCALE VALIDATION
# ============================================================================

@torch.no_grad()
def multi_scale_inference(model, images, scales=[1.0], flip=False):
    """Multi-scale inference tối ưu bộ nhớ"""
    model.eval()
    _, _, orig_h, orig_w = images.shape

    # Khởi tạo với scale đầu tiên
    scaled_imgs = F.interpolate(images, scale_factor=scales[0],
                               mode='bilinear', align_corners=False)

    with torch.cuda.amp.autocast():
        final_preds, _ = model(scaled_imgs)

    final_preds = F.interpolate(final_preds, size=(orig_h, orig_w),
                               mode='bilinear', align_corners=False)
    count = 1

    del scaled_imgs

    # Các scales còn lại
    for scale in scales[1:]:
        scaled_imgs = F.interpolate(images, scale_factor=scale,
                                   mode='bilinear', align_corners=False)

        with torch.cuda.amp.autocast():
            pred, _ = model(scaled_imgs)

        pred = F.interpolate(pred, size=(orig_h, orig_w),
                           mode='bilinear', align_corners=False)
        final_preds = final_preds + pred
        count += 1

        del scaled_imgs, pred

        # Flip augmentation
        if flip:
            scaled_imgs = F.interpolate(images, scale_factor=scale,
                                       mode='bilinear', align_corners=False)
            flipped_imgs = torch.flip(scaled_imgs, dims=[3])

            with torch.cuda.amp.autocast():
                pred_flip, _ = model(flipped_imgs)

            pred_flip = torch.flip(pred_flip, dims=[3])
            pred_flip = F.interpolate(pred_flip, size=(orig_h, orig_w),
                                    mode='bilinear', align_corners=False)
            final_preds = final_preds + pred_flip
            count += 1

            del scaled_imgs, flipped_imgs, pred_flip

    final_preds = final_preds / count
    return final_preds




# ============================================================================
# MEMORY-OPTIMIZED TRAINING FUNCTION WITH AUGMENTATION
# ============================================================================
def train_spine_curve_model(
    model,
    train_dataset,
    val_dataset,
    device='cuda',
    epochs=100,
    batch_size=4,
    lr=1e-3,
    save_path='best_spine_curve_model.pth',
    plot_save_path='training_metrics.png',
    model_segmentation_back=None,
    resume_from=None,
    # Accuracy configuration
    accuracy_thresholds=[0.05, 0.1, 0.15],
    # Augmentation configuration
    use_augmentation=True,
    p_hflip=0.5,
    p_vflip=0.3,
    p_rotate=0.5,
    rotation_range=(-15, 15),
    p_brightness=0.4,
    p_contrast=0.4,
    p_noise=0.3,
    p_blur=0.2,
    # Multiscale validation
    use_multiscale_val=False,
    multiscale_scales=[0.75, 1.0, 1.5],
    multiscale_flip=True,
    # Advanced techniques
    use_ema=False,
    use_mixup=False,
    mixup_alpha=0.2,
    # Loss configuration
    use_gradient_loss=True,
    use_peak_loss=True,
    # Memory management
    gradient_accumulation_steps=4,
    clear_cache_every_n_steps=10,
    clear_cache_after_backward=True,
    clear_cache_after_forward=False,
):
    """Training function với accuracy tracking, data augmentation và multiscale validation"""

    print(f"\n{'='*80}")
    print("🚀 TRAINING WITH ACCURACY METRICS & DATA AUGMENTATION")
    if use_multiscale_val:
        print("🔍 MULTISCALE VALIDATION ENABLED")
    print(f"{'='*80}")

    # Initial cleanup
    clear_memory(verbose=True)
    model = model.to(device)

    # Memory efficient settings
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    if torch.cuda.is_available():
        print_memory_summary(device)

    # Data augmentation
    if use_augmentation:
        augmentation = DistanceFieldAugmentation(
            p_hflip=p_hflip,
            p_vflip=p_vflip,
            p_rotate=p_rotate,
            rotation_range=rotation_range,
            p_brightness=p_brightness,
            brightness_range=(0.8, 1.2),
            p_contrast=p_contrast,
            contrast_range=(0.8, 1.2),
            p_noise=p_noise,
            noise_std=0.02,
            p_blur=p_blur
        )
        print("✅ Data augmentation enabled")
        print(f"   - Horizontal flip: {p_hflip}")
        print(f"   - Vertical flip: {p_vflip}")
        print(f"   - Rotation: {p_rotate} (range: {rotation_range}°)")
        print(f"   - Brightness: {p_brightness}")
        print(f"   - Contrast: {p_contrast}")
        print(f"   - Noise: {p_noise}")
        print(f"   - Blur: {p_blur}")
    else:
        augmentation = None
        print("❌ Data augmentation disabled")

    # Multiscale validation info
    if use_multiscale_val:
        num_scales = len(multiscale_scales)
        num_predictions = num_scales * (2 if multiscale_flip else 1)
        print(f"\n✅ Multiscale validation enabled")
        print(f"   - Scales: {multiscale_scales}")
        print(f"   - Flip: {multiscale_flip}")
        print(f"   - Total predictions per sample: {num_predictions}")
        print(f"   ⚠️  Validation will be {num_predictions}x slower")
    else:
        print(f"\n❌ Multiscale validation disabled (faster validation)")

    # Loss function
    criterion = DistanceFieldLoss(
        use_mse=True,
        use_l1=True,
        use_gradient=use_gradient_loss,
        use_peak=use_peak_loss,
        mse_weight=1.0,
        l1_weight=0.5,
        gradient_weight=0.3,
        peak_weight=0.2,
        accuracy_thresholds=accuracy_thresholds
    )

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
        betas=(0.9, 0.999)
    )

    # Scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=15, T_mult=2, eta_min=1e-6
    )

    # EMA (optional)
    ema = EMA(model, decay=0.999) if use_ema else None

    # History
    history = {
        'train_loss': [], 'val_loss': [],
        'train_mse': [], 'val_mse': [],
        'lr': []
    }

    for thresh in accuracy_thresholds:
        history[f'train_acc_{thresh:.2f}'] = []
        history[f'val_acc_{thresh:.2f}'] = []

    history['train_mae'] = []
    history['val_mae'] = []
    history['train_correlation'] = []
    history['val_correlation'] = []

    if use_gradient_loss:
        history['train_gradient'] = []
        history['val_gradient'] = []
    if use_peak_loss:
        history['train_peak'] = []
        history['val_peak'] = []

    # Training state
    start_epoch = 0
    best_val_loss = float('inf')
    best_val_acc = 0.0
    patience = 0
    max_patience = 20

    # Resume from checkpoint
    if resume_from and os.path.exists(resume_from):
        print(f"🔄 Resuming from: {resume_from}")
        checkpoint = torch.load(resume_from, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        best_val_acc = checkpoint.get('best_val_acc', 0.0)
        if 'history' in checkpoint:
            history = checkpoint['history']
        print(f"✔ Resumed from epoch {start_epoch}")
        clear_memory(verbose=True)

    # Data loaders
    print("\n📦 Creating data loaders...")
    train_loader = MemoryEfficientDataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=False, drop_last=True
    )
    val_loader = MemoryEfficientDataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=False, drop_last=False
    )

    print(f"✔ Train batches: {len(train_loader)}")
    print(f"✔ Val batches: {len(val_loader)}")
    print(f"✔ Effective batch size: {batch_size * gradient_accumulation_steps}")

    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler()

    # ========================================================================
    # TRAINING LOOP
    # ========================================================================
    for epoch in range(start_epoch, epochs):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"{'='*80}")

        # ============== TRAINING ==============
        model.train()
        train_loss = 0.0
        train_metrics = {'mse': 0, 'l1': 0, 'gradient': 0, 'peak': 0}
        train_accuracy = {f'acc_{t:.2f}': 0.0 for t in accuracy_thresholds}
        train_accuracy['mae'] = 0.0
        train_accuracy['correlation'] = 0.0

        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc="🔧 Training")

        for step, (images, distance_fields) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            distance_fields = distance_fields.to(device, non_blocking=True)
            distance_fields = torch.clamp(distance_fields, 0.0, 1.0)

            # ========== APPLY AUGMENTATION ==========
            if use_augmentation and augmentation is not None:
                # Augment từng sample trong batch
                augmented_images = []
                augmented_dfs = []

                for i in range(images.shape[0]):
                    img = images[i]
                    df = distance_fields[i]

                    # Apply augmentation
                    img_aug, df_aug = augmentation(img, df)

                    augmented_images.append(img_aug)
                    augmented_dfs.append(df_aug)

                images = torch.stack(augmented_images)
                distance_fields = torch.stack(augmented_dfs)

            # Background masking (optional)
            if model_segmentation_back is not None:
                with torch.no_grad():
                    mask = model_segmentation_back.predict(image_tensor=images)
                    if mask.dim() == 3:
                        mask = mask.unsqueeze(1)
                    if mask.shape[-2:] != images.shape[-2:]:
                        mask = F.interpolate(mask, size=images.shape[-2:],
                                           mode="bilinear", align_corners=False)
                    images = images * mask
                    del mask

            # Forward pass
            with torch.cuda.amp.autocast():
                pred, deep_sup = model(images)

                if deep_sup and len(deep_sup) > 2:
                    deep_sup = deep_sup[:2]

                # Compute loss and accuracy
                loss, loss_dict, accuracy_dict = criterion(
                    pred, distance_fields,
                    deep_sup_preds=deep_sup,
                    compute_accuracy=True
                )
                loss = loss / gradient_accumulation_steps

            if clear_cache_after_forward:
                del pred, deep_sup
                clear_memory()

            # Backward pass
            scaler.scale(loss).backward()

            if clear_cache_after_backward:
                clear_memory()

            # Update weights
            if (step + 1) % gradient_accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

                if ema is not None:
                    ema.update()

            # Accumulate metrics
            train_loss += loss.item() * gradient_accumulation_steps
            for k in ['mse', 'l1', 'gradient', 'peak']:
                train_metrics[k] += loss_dict.get(k, 0)

            for k, v in accuracy_dict.items():
                train_accuracy[k] += v

            # Update progress bar
            pbar.set_postfix({
                'loss': f"{loss.item() * gradient_accumulation_steps:.4f}",
                'acc': f"{accuracy_dict.get('acc_0.10', 0):.3f}"
            })

            # Periodic cache clearing
            if (step + 1) % clear_cache_every_n_steps == 0:
                clear_memory()

        # Average training metrics
        n = len(train_loader)
        train_loss /= n
        for k in train_metrics:
            train_metrics[k] /= n
        for k in train_accuracy:
            train_accuracy[k] /= n

        # Cleanup after training
        del images, distance_fields, loss, loss_dict, accuracy_dict
        clear_memory(verbose=True)

        # ============== VALIDATION (NO AUGMENTATION) ==============
        model.eval()
        val_loss = 0.0
        val_metrics = {'mse': 0, 'l1': 0, 'gradient': 0, 'peak': 0}
        val_accuracy = {f'acc_{t:.2f}': 0.0 for t in accuracy_thresholds}
        val_accuracy['mae'] = 0.0
        val_accuracy['correlation'] = 0.0

        with torch.no_grad():
            val_desc = "🔍 Validation (Multiscale)" if use_multiscale_val else "🔍 Validation"
            for batch_idx, (images, distance_fields) in enumerate(tqdm(val_loader, desc=val_desc)):
                images = images.to(device, non_blocking=True)
                distance_fields = distance_fields.to(device, non_blocking=True)
                distance_fields = torch.clamp(distance_fields, 0.0, 1.0)

                with torch.cuda.amp.autocast():
                    # Sử dụng multiscale prediction nếu enabled
                    if use_multiscale_val:
                        pred = multi_scale_inference(
                            model, images,
                            scales=multiscale_scales,
                            flip=multiscale_flip,
                            device=device
                        )
                    else:
                        # Normal single-scale prediction
                        pred, _ = model(images)

                    loss, loss_dict, accuracy_dict = criterion(
                        pred, distance_fields,
                        deep_sup_preds=None,
                        compute_accuracy=True
                    )

                val_loss += loss.item()
                for k in ['mse', 'l1', 'gradient', 'peak']:
                    val_metrics[k] += loss_dict.get(k, 0)

                for k, v in accuracy_dict.items():
                    val_accuracy[k] += v

                del pred, loss, loss_dict, accuracy_dict

                if (batch_idx + 1) % 5 == 0:
                    clear_memory()

        # Average validation metrics
        m = len(val_loader)
        val_loss /= m
        for k in val_metrics:
            val_metrics[k] /= m
        for k in val_accuracy:
            val_accuracy[k] /= m

        # Cleanup after validation
        del images, distance_fields
        clear_memory(verbose=True)

        # Update scheduler
        lr_current = optimizer.param_groups[0]['lr']
        scheduler.step()

        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_mse'].append(train_metrics['mse'])
        history['val_mse'].append(val_metrics['mse'])
        history['lr'].append(lr_current)

        for thresh in accuracy_thresholds:
            key = f'acc_{thresh:.2f}'
            history[f'train_{key}'].append(train_accuracy[key])
            history[f'val_{key}'].append(val_accuracy[key])

        history['train_mae'].append(train_accuracy['mae'])
        history['val_mae'].append(val_accuracy['mae'])
        history['train_correlation'].append(train_accuracy['correlation'])
        history['val_correlation'].append(val_accuracy['correlation'])

        if use_gradient_loss:
            history['train_gradient'].append(train_metrics['gradient'])
            history['val_gradient'].append(val_metrics['gradient'])
        if use_peak_loss:
            history['train_peak'].append(train_metrics['peak'])
            history['val_peak'].append(val_metrics['peak'])

        # ============== LOGGING ==============
        print(f"\n📊 Results:")
        print(f"Loss    | Train: {train_loss:.4f} | Val: {val_loss:.4f}")
        print(f"MSE     | Train: {train_metrics['mse']:.4f} | Val: {val_metrics['mse']:.4f}")
        print(f"MAE     | Train: {train_accuracy['mae']:.4f} | Val: {val_accuracy['mae']:.4f}")

        for thresh in accuracy_thresholds:
            key = f'acc_{thresh:.2f}'
            print(f"Acc<{thresh:.2f} | Train: {train_accuracy[key]:.4f} | Val: {val_accuracy[key]:.4f}")

        print(f"Corr    | Train: {train_accuracy['correlation']:.4f} | Val: {val_accuracy['correlation']:.4f}")
        print(f"LR      | {lr_current:.6f}")

        if torch.cuda.is_available():
            current = torch.cuda.memory_allocated(device) / 1024**3
            peak = torch.cuda.max_memory_allocated(device) / 1024**3
            print(f"GPU     | Current={current:.2f}GB | Peak={peak:.2f}GB")

        # ============== CHECKPOINTING ==============
        current_val_acc = val_accuracy['acc_0.10']

        if val_loss < best_val_loss or current_val_acc > best_val_acc:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                print(f"✅ Best val loss: {best_val_loss:.4f}")

            if current_val_acc > best_val_acc:
                best_val_acc = current_val_acc
                print(f"✅ Best val accuracy: {best_val_acc:.4f}")

            patience = 0

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_loss': val_loss,
                'best_val_loss': best_val_loss,
                'val_accuracy': current_val_acc,
                'best_val_acc': best_val_acc,
                'history': history,
                'config': {
                    'batch_size': batch_size,
                    'gradient_accumulation_steps': gradient_accumulation_steps,
                    'accuracy_thresholds': accuracy_thresholds,
                    'use_augmentation': use_augmentation,
                    'use_multiscale_val': use_multiscale_val,
                    'multiscale_scales': multiscale_scales,
                    'multiscale_flip': multiscale_flip,
                }
            }

            torch.save(checkpoint, save_path)
            print("💾 Best model saved!")
        else:
            patience += 1
            print(f"⏳ Patience: {patience}/{max_patience}")

        # Early stopping
        if patience >= max_patience:
            print("\n⛔ Early stopping!")
            break

        # Cleanup after epoch
        torch.cuda.reset_peak_memory_stats(device)
        clear_memory(verbose=True)
        print_memory_summary(device)
    # Plot metrics
    try:
        plot_training_metrics(history, plot_save_path)
    except:
        pass

    # Final summary
    print("\n" + "="*80)
    print("🎉 Training Complete!")
    print(f"📁 Best model: {save_path}")
    print(f"📉 Best val loss: {best_val_loss:.4f}")
    print(f"🎯 Best val accuracy: {best_val_acc:.4f}")
    print("="*80)

    clear_memory(verbose=True)

    return history


def plot_training_metrics(history, save_path='training_metrics.png', figsize=(20, 12), dpi=100):
    import matplotlib.pyplot as plt
    import numpy as np

    # Thiết lập style
    plt.style.use('seaborn-v0_8-darkgrid')

    # Tính số epoch
    epochs = range(1, len(history['train_loss']) + 1)

    # Tạo figure với nhiều subplots
    fig = plt.figure(figsize=figsize, dpi=dpi)

    # ========== 1. Loss Curves ==========
    ax1 = plt.subplot(3, 3, 1)
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
    ax1.plot(epochs, history['val_loss'], 'r-', linewidth=2, label='Val Loss')
    ax1.set_xlabel('Epoch', fontsize=10)
    ax1.set_ylabel('Loss', fontsize=10)
    ax1.set_title('📉 Total Loss', fontsize=12, fontweight='bold')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)

    # ========== 2. MSE ==========
    ax2 = plt.subplot(3, 3, 2)
    ax2.plot(epochs, history['train_mse'], 'b-', linewidth=2, label='Train MSE')
    ax2.plot(epochs, history['val_mse'], 'r-', linewidth=2, label='Val MSE')
    ax2.set_xlabel('Epoch', fontsize=10)
    ax2.set_ylabel('MSE', fontsize=10)
    ax2.set_title('📊 Mean Squared Error', fontsize=12, fontweight='bold')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    # ========== 3. MAE ==========
    ax3 = plt.subplot(3, 3, 3)
    ax3.plot(epochs, history['train_mae'], 'b-', linewidth=2, label='Train MAE')
    ax3.plot(epochs, history['val_mae'], 'r-', linewidth=2, label='Val MAE')
    ax3.set_xlabel('Epoch', fontsize=10)
    ax3.set_ylabel('MAE', fontsize=10)
    ax3.set_title('📏 Mean Absolute Error', fontsize=12, fontweight='bold')
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)

    # ========== 4. Accuracy Curves (tất cả thresholds) ==========
    ax4 = plt.subplot(3, 3, 4)

    # Tìm tất cả accuracy thresholds
    acc_keys = [k for k in history.keys() if k.startswith('train_acc_')]

    colors = plt.cm.tab10(np.linspace(0, 1, len(acc_keys)))

    for i, key in enumerate(acc_keys):
        thresh = key.replace('train_acc_', '')
        val_key = f'val_acc_{thresh}'

        ax4.plot(epochs, history[key], '--', linewidth=1.5,
                color=colors[i], alpha=0.7, label=f'Train <{thresh}')
        ax4.plot(epochs, history[val_key], '-', linewidth=2,
                color=colors[i], label=f'Val <{thresh}')

    ax4.set_xlabel('Epoch', fontsize=10)
    ax4.set_ylabel('Accuracy', fontsize=10)
    ax4.set_title('🎯 Accuracy at Different Thresholds', fontsize=12, fontweight='bold')
    ax4.legend(loc='lower right', fontsize=8)
    ax4.grid(True, alpha=0.3)
    ax4.set_ylim([0, 1])

    # ========== 5. Correlation ==========
    ax5 = plt.subplot(3, 3, 5)
    ax5.plot(epochs, history['train_correlation'], 'b-', linewidth=2,
            label='Train Correlation')
    ax5.plot(epochs, history['val_correlation'], 'r-', linewidth=2,
            label='Val Correlation')
    ax5.set_xlabel('Epoch', fontsize=10)
    ax5.set_ylabel('Correlation', fontsize=10)
    ax5.set_title('🔗 Prediction Correlation', fontsize=12, fontweight='bold')
    ax5.legend(loc='lower right')
    ax5.grid(True, alpha=0.3)
    ax5.set_ylim([0, 1])

    # ========== 6. Learning Rate ==========
    ax6 = plt.subplot(3, 3, 6)
    ax6.plot(epochs, history['lr'], 'g-', linewidth=2)
    ax6.set_xlabel('Epoch', fontsize=10)
    ax6.set_ylabel('Learning Rate', fontsize=10)
    ax6.set_title('📈 Learning Rate Schedule', fontsize=12, fontweight='bold')
    ax6.set_yscale('log')
    ax6.grid(True, alpha=0.3)

    # ========== 7. Gradient Loss (nếu có) ==========
    if 'train_gradient' in history:
        ax7 = plt.subplot(3, 3, 7)
        ax7.plot(epochs, history['train_gradient'], 'b-', linewidth=2,
                label='Train Gradient')
        ax7.plot(epochs, history['val_gradient'], 'r-', linewidth=2,
                label='Val Gradient')
        ax7.set_xlabel('Epoch', fontsize=10)
        ax7.set_ylabel('Gradient Loss', fontsize=10)
        ax7.set_title('🌊 Gradient Loss', fontsize=12, fontweight='bold')
        ax7.legend(loc='upper right')
        ax7.grid(True, alpha=0.3)

    # ========== 8. Peak Loss (nếu có) ==========
    if 'train_peak' in history:
        ax8 = plt.subplot(3, 3, 8)
        ax8.plot(epochs, history['train_peak'], 'b-', linewidth=2,
                label='Train Peak')
        ax8.plot(epochs, history['val_peak'], 'r-', linewidth=2,
                label='Val Peak')
        ax8.set_xlabel('Epoch', fontsize=10)
        ax8.set_ylabel('Peak Loss', fontsize=10)
        ax8.set_title('⛰️ Peak Loss', fontsize=12, fontweight='bold')
        ax8.legend(loc='upper right')
        ax8.grid(True, alpha=0.3)

    # ========== 9. Best Accuracy Comparison ==========
    ax9 = plt.subplot(3, 3, 9)

    # Lấy accuracy tốt nhất cho mỗi threshold
    best_train_acc = []
    best_val_acc = []
    threshold_labels = []

    for key in acc_keys:
        thresh = key.replace('train_acc_', '')
        val_key = f'val_acc_{thresh}'

        best_train_acc.append(max(history[key]))
        best_val_acc.append(max(history[val_key]))
        threshold_labels.append(f'<{thresh}')

    x = np.arange(len(threshold_labels))
    width = 0.35

    ax9.bar(x - width/2, best_train_acc, width, label='Train',
           color='steelblue', alpha=0.8)
    ax9.bar(x + width/2, best_val_acc, width, label='Val',
           color='tomato', alpha=0.8)

    ax9.set_xlabel('Threshold', fontsize=10)
    ax9.set_ylabel('Best Accuracy', fontsize=10)
    ax9.set_title('🏆 Best Accuracy Comparison', fontsize=12, fontweight='bold')
    ax9.set_xticks(x)
    ax9.set_xticklabels(threshold_labels)
    ax9.legend()
    ax9.grid(True, alpha=0.3, axis='y')
    ax9.set_ylim([0, 1])

    # Thêm giá trị trên mỗi bar
    for i, v in enumerate(best_train_acc):
        ax9.text(i - width/2, v + 0.02, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)
    for i, v in enumerate(best_val_acc):
        ax9.text(i + width/2, v + 0.02, f'{v:.3f}',
                ha='center', va='bottom', fontsize=8)

    # ========== Tổng kết metrics ==========
    fig.text(0.5, 0.02,
             f'Final: Train Loss={history["train_loss"][-1]:.4f} | '
             f'Val Loss={history["val_loss"][-1]:.4f} | '
             f'Best Val Loss={min(history["val_loss"]):.4f} | '
             f'Val MAE={history["val_mae"][-1]:.4f}',
             ha='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.suptitle('📊 Training Metrics Dashboard',
                fontsize=16, fontweight='bold', y=0.995)

    plt.tight_layout(rect=[0, 0.03, 1, 0.99])

    # Lưu figure
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"✅ Metrics plot saved to: {save_path}")

    plt.show()

    # In summary
    print("\n" + "="*80)
    print("📊 TRAINING SUMMARY")
    print("="*80)
    print(f"Total Epochs: {len(epochs)}")
    print(f"\n🔴 Best Validation Metrics:")
    print(f"   Loss:        {min(history['val_loss']):.6f} (epoch {np.argmin(history['val_loss'])+1})")
    print(f"   MSE:         {min(history['val_mse']):.6f}")
    print(f"   MAE:         {min(history['val_mae']):.6f}")

    for key in acc_keys:
        thresh = key.replace('train_acc_', '')
        val_key = f'val_acc_{thresh}'
        best_acc = max(history[val_key])
        best_epoch = np.argmax(history[val_key]) + 1
        print(f"   Acc <{thresh}:   {best_acc:.6f} (epoch {best_epoch})")

    print(f"   Correlation: {max(history['val_correlation']):.6f}")

    print(f"\n🔵 Final Training Metrics:")
    print(f"   Loss:        {history['train_loss'][-1]:.6f}")
    print(f"   MSE:         {history['train_mse'][-1]:.6f}")
    print(f"   MAE:         {history['train_mae'][-1]:.6f}")

    for key in acc_keys:
        thresh = key.replace('train_acc_', '')
        print(f"   Acc <{thresh}:   {history[key][-1]:.6f}")

    print(f"   Correlation: {history['train_correlation'][-1]:.6f}")
    print("="*80)


# ========== HÀM BỔ SUNG: Plot chi tiết từng metric ==========
def plot_detailed_metrics(history, save_dir='metrics_plots', dpi=100):
    """
    Vẽ chi tiết từng metric riêng biệt

    Args:
        history: dict chứa training metrics
        save_dir: thư mục lưu các plots
        dpi: độ phân giải
    """
    import os
    os.makedirs(save_dir, exist_ok=True)

    epochs = range(1, len(history['train_loss']) + 1)

    # 1. Loss plot chi tiết
    plt.figure(figsize=(10, 6), dpi=dpi)
    plt.plot(epochs, history['train_loss'], 'b-', linewidth=2,
            label='Train Loss', marker='o', markersize=3)
    plt.plot(epochs, history['val_loss'], 'r-', linewidth=2,
            label='Val Loss', marker='s', markersize=3)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/loss_curve.png', dpi=dpi, bbox_inches='tight')
    plt.close()

    # 2. Accuracy plot chi tiết
    acc_keys = [k for k in history.keys() if k.startswith('train_acc_')]

    plt.figure(figsize=(12, 6), dpi=dpi)
    colors = plt.cm.tab10(np.linspace(0, 1, len(acc_keys)))

    for i, key in enumerate(acc_keys):
        thresh = key.replace('train_acc_', '')
        val_key = f'val_acc_{thresh}'

        plt.plot(epochs, history[key], '--', linewidth=1.5,
                color=colors[i], alpha=0.6, label=f'Train <{thresh}')
        plt.plot(epochs, history[val_key], '-', linewidth=2,
                color=colors[i], marker='o', markersize=2, label=f'Val <{thresh}')

    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Accuracy', fontsize=12)
    plt.title('Accuracy at Different Thresholds', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 1])
    plt.tight_layout()
    plt.savefig(f'{save_dir}/accuracy_curves.png', dpi=dpi, bbox_inches='tight')
    plt.close()

    print(f"✅ Detailed plots saved to: {save_dir}/")


# ========== HÀM BỔ SUNG: So sánh nhiều runs ==========
def compare_training_runs(histories_dict, save_path='comparison.png',
                         metric='val_loss', dpi=100):
    """
    So sánh nhiều training runs

    Args:
        histories_dict: dict of {run_name: history}
        save_path: đường dẫn lưu figure
        metric: metric để so sánh (e.g., 'val_loss', 'val_acc_0.10')
        dpi: độ phân giải
    """
    plt.figure(figsize=(12, 6), dpi=dpi)

    colors = plt.cm.tab10(np.linspace(0, 1, len(histories_dict)))

    for i, (name, history) in enumerate(histories_dict.items()):
        epochs = range(1, len(history[metric]) + 1)
        plt.plot(epochs, history[metric], linewidth=2,
                color=colors[i], marker='o', markersize=3, label=name)

    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel(metric.replace('_', ' ').title(), fontsize=12)
    plt.title(f'Comparison: {metric.replace("_", " ").title()}',
             fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
    print(f"✅ Comparison plot saved to: {save_path}")
    plt.show()


    clear_memory(verbose=True)

    return history


# ============================================================================
# TEST SET EVALUATION
# ============================================================================
@torch.no_grad()
def evaluate_on_test_set(
    model,
    test_dataset,
    device='cuda',
    batch_size=4,
    checkpoint_path=None,
    use_multiscale=False,
    multiscale_scales=[0.75, 1.0, 1.5],
    multiscale_flip=True,
    accuracy_thresholds=[0.05, 0.1, 0.15],
    save_predictions=False,
    predictions_dir='test_predictions',
    num_visualize=5
):
    """
    Đánh giá model trên test set và tính toán các metrics chi tiết

    Args:
        model: model cần đánh giá
        test_dataset: test dataset
        device: device để chạy
        batch_size: batch size
        checkpoint_path: đường dẫn checkpoint để load (nếu có)
        use_multiscale: có dùng multiscale inference không
        multiscale_scales: các scales cho multiscale
        multiscale_flip: có dùng flip augmentation không
        accuracy_thresholds: các ngưỡng để tính accuracy
        save_predictions: có lưu predictions không
        predictions_dir: thư mục lưu predictions
        num_visualize: số lượng samples để visualize

    Returns:
        test_metrics: dict chứa các metrics
    """
    print(f"\n{'='*80}")
    print("🧪 EVALUATING ON TEST SET")
    print(f"{'='*80}")

    # Load checkpoint nếu có
    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"📦 Loading checkpoint: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✅ Checkpoint loaded successfully")

    model = model.to(device)
    model.eval()

    # Tạo thư mục lưu predictions
    if save_predictions:
        os.makedirs(predictions_dir, exist_ok=True)
        print(f"📁 Predictions will be saved to: {predictions_dir}")

    # Khởi tạo loss function
    criterion = DistanceFieldLoss(
        use_mse=True,
        use_l1=True,
        use_gradient=True,
        use_peak=True,
        mse_weight=1.0,
        l1_weight=0.5,
        gradient_weight=0.3,
        peak_weight=0.2,
        accuracy_thresholds=accuracy_thresholds
    )

    # Tạo dataloader
    test_loader = MemoryEfficientDataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=False,
        drop_last=False
    )

    print(f"📊 Test set size: {len(test_dataset)}")
    print(f"📦 Number of batches: {len(test_loader)}")
    if use_multiscale:
        print(f"🔍 Using multiscale inference: {multiscale_scales}")
        print(f"🔄 Using flip augmentation: {multiscale_flip}")

    # Initialize metrics
    test_loss = 0.0
    test_metrics = {'mse': 0, 'l1': 0, 'gradient': 0, 'peak': 0}
    test_accuracy = {f'acc_{t:.2f}': 0.0 for t in accuracy_thresholds}
    test_accuracy['mae'] = 0.0
    test_accuracy['correlation'] = 0.0

    # Lưu predictions để visualize
    visualization_samples = []

    # Evaluate
    desc = "🧪 Testing (Multiscale)" if use_multiscale else "🧪 Testing"
    for batch_idx, (images, distance_fields) in enumerate(tqdm(test_loader, desc=desc)):
        images = images.to(device, non_blocking=True)
        distance_fields = distance_fields.to(device, non_blocking=True)
        distance_fields = torch.clamp(distance_fields, 0.0, 1.0)

        with torch.cuda.amp.autocast():
            # Prediction
            if use_multiscale:
                pred = multi_scale_inference(
                    model, images,
                    scales=multiscale_scales,
                    flip=multiscale_flip
                )
            else:
                pred, _ = model(images)

            # Compute metrics
            loss, loss_dict, accuracy_dict = criterion(
                pred, distance_fields,
                deep_sup_preds=None,
                compute_accuracy=True
            )

        # Accumulate metrics
        test_loss += loss.item()
        for k in ['mse', 'l1', 'gradient', 'peak']:
            test_metrics[k] += loss_dict.get(k, 0)

        for k, v in accuracy_dict.items():
            test_accuracy[k] += v

        # Lưu samples để visualize
        if len(visualization_samples) < num_visualize:
            for i in range(min(images.shape[0], num_visualize - len(visualization_samples))):
                visualization_samples.append({
                    'image': images[i].cpu(),
                    'ground_truth': distance_fields[i].cpu(),
                    'prediction': pred[i].cpu()
                })

        # Lưu predictions nếu cần
        if save_predictions:
            for i in range(images.shape[0]):
                idx = batch_idx * batch_size + i
                pred_np = pred[i].cpu().numpy()
                gt_np = distance_fields[i].cpu().numpy()

                np.save(
                    os.path.join(predictions_dir, f'pred_{idx:04d}.npy'),
                    pred_np
                )
                np.save(
                    os.path.join(predictions_dir, f'gt_{idx:04d}.npy'),
                    gt_np
                )

        del pred, loss, loss_dict, accuracy_dict

        if (batch_idx + 1) % 5 == 0:
            clear_memory()

    # Average metrics
    n = len(test_loader)
    test_loss /= n
    for k in test_metrics:
        test_metrics[k] /= n
    for k in test_accuracy:
        test_accuracy[k] /= n

    # Cleanup
    del images, distance_fields
    clear_memory(verbose=True)

    # Print results
    print(f"\n{'='*80}")
    print("📊 TEST SET RESULTS")
    print(f"{'='*80}")
    print(f"Total Loss:     {test_loss:.6f}")
    print(f"MSE:            {test_metrics['mse']:.6f}")
    print(f"L1:             {test_metrics['l1']:.6f}")
    print(f"Gradient Loss:  {test_metrics['gradient']:.6f}")
    print(f"Peak Loss:      {test_metrics['peak']:.6f}")
    print(f"\n📏 Error Metrics:")
    print(f"MAE:            {test_accuracy['mae']:.6f}")
    print(f"\n🎯 Accuracy Metrics:")
    for thresh in accuracy_thresholds:
        key = f'acc_{thresh:.2f}'
        print(f"Accuracy <{thresh:.2f}:  {test_accuracy[key]:.6f} ({test_accuracy[key]*100:.2f}%)")
    print(f"\n🔗 Correlation:   {test_accuracy['correlation']:.6f}")
    print(f"{'='*80}")

    # Visualize results


    # Prepare return dict
    results = {
        'loss': test_loss,
        'mse': test_metrics['mse'],
        'l1': test_metrics['l1'],
        'gradient': test_metrics['gradient'],
        'peak': test_metrics['peak'],
        'mae': test_accuracy['mae'],
        'correlation': test_accuracy['correlation']
    }

    for thresh in accuracy_thresholds:
        key = f'acc_{thresh:.2f}'
        results[key] = test_accuracy[key]

    return results



k=fold

# PRE PROCESSING

In [ ]:
# @title

# ============================================================================
# PRE PROCESSING
# ============================================================================


class PreProcessing:
    """
    Image pre-processing module before feeding the image to the model.
    Includes:
        - Gamma correction
        - CLAHE
        - Denoising
        - Normalization
    """

    # ------------------------------ CHECK INPUT ------------------------------
    def _validate(self, image, func_name=""):
        if image is None:
            raise ValueError(f"[PreProcessing ERROR] Input image is None before {func_name}.")
        if not isinstance(image, np.ndarray):
            raise TypeError(f"[PreProcessing ERROR] Expected numpy array in {func_name}.")
        return True

    # ------------------------------ LOAD IMAGE ------------------------------
    def load_image(self, image_input):
        """
        Accept either a numpy array or an image path.
        """
        if isinstance(image_input, str):
            if not os.path.exists(image_input):
                raise FileNotFoundError(f"[PreProcessing ERROR] Image path not found: {image_input}")
            image = cv2.imread(image_input)
            if image is None:
                raise ValueError(f"[PreProcessing ERROR] Cannot read image: {image_input}")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            return image
        elif isinstance(image_input, np.ndarray):
            return image_input
        else:
            raise TypeError(f"[PreProcessing ERROR] Input must be a numpy array or image path.")

    # ------------------------------ 1. GAMMA CORRECTION ------------------------------
    def auto_gamma_correction(self, image, target_brightness=0.5):
        self._validate(image, "auto_gamma_correction")

        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        brightness = max(gray.mean() / 255.0, 1e-4)
        gamma = np.log(target_brightness) / np.log(brightness)

        corrected = np.power(image / 255.0, gamma)
        corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
        return corrected

    # ------------------------------ 2. CLAHE ------------------------------
    def apply_CLAHE(self, image):
        self._validate(image, "apply_CLAHE")

        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        l2 = clahe.apply(l)

        lab = cv2.merge([l2, a, b])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # ------------------------------ 3. DENOISE ------------------------------
    def denoise_preserve_edges(self, image):
        self._validate(image, "denoise_preserve_edges")
        return cv2.bilateralFilter(image, d=7, sigmaColor=50, sigmaSpace=50)

    # ------------------------------ 4. NORMALIZE ------------------------------
    def normalize(self, image):
        self._validate(image, "normalize")
        return image.astype(np.float32) / 255.0

    # ------------------------------ 5. FULL PIPELINE ------------------------------
    def process(self, image_input):
        """
        Safe pre-processing pipeline.
        Accepts either image path or numpy array.
        Returns normalized image (float32).
        """
        try:
            image = self.load_image(image_input)
            img = self.auto_gamma_correction(image)
            img = self.apply_CLAHE(img)
            img = self.denoise_preserve_edges(img)
            # img = self.normalize(img)
            return img

        except Exception as e:
            raise RuntimeError(f"[PreProcessing] Error during pipeline: {str(e)}")




# ============================================================================
# DYNAMIC AUGMENTATION SCHEDULER
# ============================================================================


class DynamicAugmentationScheduler:
    """Random augmentation parameters each epoch (Improved Version)"""

    def __init__(self, image_size=512):
        self.image_size = image_size
        self.current_epoch = 0

    def get_augmentation_for_epoch(self, epoch, is_train=True):
        self.current_epoch = epoch

        if not is_train:
            return self._get_val_transform()

        return self._get_random_augmentation()

    # =========================
    # 🔥 IMPROVED AUGMENTATION
    # =========================
    def _get_random_augmentation(self):

        # Soft geometric params (giảm sai lệch keypoints)
        shift = random.uniform(0.01, 0.08)
        scale = random.uniform(0.03, 0.18)
        rotate = random.randint(3, 18)

        # Soft brightness/contrast
        br_limit = random.uniform(0.05, 0.4)
        ct_limit = random.uniform(0.05, 0.4)

        # Noise control
        noise_min = random.randint(3, 12)
        noise_max = random.randint(12, 25)

        # Blur control
        blur_limit = random.choice([(3, 5), (3, 7)])

        # Coarse dropout small
        holes = random.randint(1, 4)
        max_hw = random.randint(20, 45)
        min_hw = random.randint(10, 20)

        # Safer geometric transformations
        geometric = A.OneOf([
            A.ShiftScaleRotate(
                shift_limit=shift,
                scale_limit=scale,
                rotate_limit=rotate,
                border_mode=cv2.BORDER_REFLECT,
                p=1.0
            ),
            A.Affine(
                shear=random.uniform(-10, 10),
                rotate=rotate,
                translate_percent=shift,
                scale=(1 - scale, 1 + scale),
                p=1.0
            )
        ], p=0.55)

        # Color transformation improved
        color_transform = A.OneOf([
            A.RandomBrightnessContrast(
            brightness_limit=(-br_limit, br_limit),
            contrast_limit=(-ct_limit, ct_limit)),
            A.CLAHE(clip_limit=random.uniform(2, 4), p=1.0),
            A.RandomGamma(gamma_limit=(80, 120), p=1.0),
            A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=1.0),
            A.ChannelShuffle(p=1.0)
        ], p=0.5)

        # Sharpen - tăng feature edges cho mô hình
        sharpen = A.OneOf([
            A.Sharpen(alpha=(0.1, 0.4), lightness=(0.8, 1.2), p=1.0),
            A.UnsharpMask(radius=3, p=1.0)
        ], p=0.35)

        # MixUp / CutMix nhẹ (an toàn landmark, áp dụng p thấp)
        mix_regularization = A.OneOf([
            A.MultiplicativeNoise(multiplier=(0.8, 1.2), p=1.0),
            A.RandomToneCurve(scale=0.1, p=1.0)
        ], p=0.25)

        return A.Compose([
            A.Resize(self.image_size, self.image_size),

            geometric,
            color_transform,
            sharpen,
            mix_regularization,

            A.OneOf([
                A.GaussNoise(var_limit=(noise_min, noise_max), p=1.0),
                A.GaussianBlur(blur_limit=blur_limit, p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.35),

            A.CoarseDropout(
                max_holes=holes,
                max_height=max_hw,
                max_width=max_hw,
                min_holes=max(1, holes // 2),
                min_height=min_hw,
                min_width=min_hw,
                p=0.25
            ),

            # Shadow soft (giảm bóng đổ cực mạnh)
            A.RandomShadow(num_shadows_lower=1, num_shadows_upper=2,
                           shadow_dimension=4, p=0.15),

            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),

            ToTensorV2()
        ], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))

    # Validation Transform
    def _get_val_transform(self):
        return A.Compose([
            A.Resize(self.image_size, self.image_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ], keypoint_params=A.KeypointParams(format='xy', remove_invisible=False))



# DATASET

In [ ]:
# @title
class SpineCurveDataset(Dataset):
    """Dataset for spine curve detection from COCO landmarks with enhanced augmentation"""

    def __init__(self, image_dir: str, coco_file: str,
                 augmentation_scheduler: DynamicAugmentationScheduler,
                 is_train: bool = True,
                 curve_thickness: int = 10,
                 max_distance: int = 10,
                 num_curve_points: int = 200,
                 # Enhanced augmentation parameters
                 brightness_limit: float = 0.3,
                 contrast_limit: float = 0.3,
                 rotation_limit: int = 15):

        self.image_dir = Path(image_dir)
        self.is_train = is_train
        self.aug_scheduler = augmentation_scheduler
        self.curve_thickness = curve_thickness
        self.num_curve_points = num_curve_points
        self.max_distance = max_distance

        # Store augmentation limits
        self.brightness_limit = brightness_limit
        self.contrast_limit = contrast_limit
        self.rotation_limit = rotation_limit

        # Load COCO
        self.coco = COCO(coco_file)
        self.image_ids = list(self.coco.imgs.keys())

        # Get number of keypoints
        categories = self.coco.loadCats(self.coco.getCatIds())
        if categories and 'keypoints' in categories[0]:
            self.num_keypoints = len(categories[0]['keypoints'])
        else:
            ann = self.coco.loadAnns(self.coco.getAnnIds(imgIds=self.image_ids[0]))[0]
            self.num_keypoints = len(ann['keypoints']) // 3

        print(f"Dataset initialized: {len(self.image_ids)} images, {self.num_keypoints} keypoints")
        print(f"Enhanced augmentation - Brightness: ±{brightness_limit}, Contrast: ±{contrast_limit}, Rotation: ±{rotation_limit}°")

    def get_enhanced_augmentation(self, epoch: int):
        """Get augmentation with progressive difficulty"""
        image_size = self.aug_scheduler.image_size

        if not self.is_train:
            # Validation: no augmentation
            return A.Compose([
                A.Resize(image_size, image_size),
                A.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ], is_check_shapes=False)

        # Progressive augmentation strength based on epoch
        progress = min(epoch / 30.0, 1.0)  # Reach full strength at epoch 30

        brightness = self.brightness_limit * progress
        contrast = self.contrast_limit * progress
        rotation = self.rotation_limit * progress

        return A.Compose([
            # RESIZE FIRST to ensure same dimensions
            A.Resize(image_size, image_size, interpolation=cv2.INTER_LINEAR),

            # Geometric transformations
            A.Rotate(
                limit=rotation,
                interpolation=cv2.INTER_LINEAR,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.7
            ),
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.15,
                rotate_limit=0,  # Already handled by Rotate
                interpolation=cv2.INTER_LINEAR,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.5
            ),

            # Flip
            A.HorizontalFlip(p=0.5),

            # Photometric transformations
            A.RandomBrightnessContrast(
                brightness_limit=brightness,
                contrast_limit=contrast,
                brightness_by_max=True,
                p=0.8
            ),

            # Additional color augmentations
            A.HueSaturationValue(
                hue_shift_limit=10,
                sat_shift_limit=20,
                val_shift_limit=15,
                p=0.5
            ),

            # Simulate different lighting conditions
            A.RandomGamma(
                gamma_limit=(80, 120),
                p=0.5
            ),

            # Simulate noise and blur (medical imaging artifacts)
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.3),

            # CLAHE for local contrast enhancement
            A.CLAHE(
                clip_limit=2.0,
                tile_grid_size=(8, 8),
                p=0.3
            ),

            # Final normalize
            A.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ], is_check_shapes=False)

    def set_epoch(self, epoch):
        """Update augmentation for new epoch"""
        self.current_epoch = epoch

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # Load image
        img_id = self.image_ids[idx]
        img_info = self.coco.loadImgs(img_id)[0]
        img_path = self.image_dir / img_info['file_name']

        image = cv2.imread(str(img_path))
        if image is None:
            return torch.zeros(3, CONFIG['image_size'], CONFIG['image_size']), torch.zeros(1, 128, 128)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        orig_h, orig_w = image.shape[:2]

        # Load landmarks
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        if anns and 'keypoints' in anns[0]:
            keypoints = np.array(anns[0]['keypoints']).reshape(-1, 3)
        else:
            keypoints = np.zeros((self.num_keypoints, 3), dtype=np.float32)

        landmarks = keypoints[:, :2].astype(np.float32)
        visibility = keypoints[:, 2]
        landmarks[visibility == 0] = np.nan

        # Fit smooth spine curve
        curve_points = SpineCurveGenerator.fit_smooth_curve(
            landmarks,
            num_points=self.num_curve_points,
            smoothing=0.5
        )

        # Generate distance field instead of binary mask
        if curve_points is not None:
            distance_field = SpineCurveGenerator.generate_distance_field(
                curve_points,
                (orig_h, orig_w),
                max_distance=self.max_distance
            )
            # Ensure float32
            distance_field = distance_field.astype(np.float32)
        else:
            distance_field = np.zeros((orig_h, orig_w), dtype=np.float32)

        # Get enhanced augmentation
        current_epoch = getattr(self, 'current_epoch', 0)
        transform = self.get_enhanced_augmentation(current_epoch)

        image_size = self.aug_scheduler.image_size

        try:
            # Resize BOTH image and distance field to same size FIRST
            image_resized = cv2.resize(image, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
            dist_resized = cv2.resize(distance_field, (image_size, image_size), interpolation=cv2.INTER_LINEAR)

            # Ensure float32 for distance field
            dist_resized = dist_resized.astype(np.float32)

            # Now apply augmentation to both (they have same size)
            transformed = transform(image=image_resized, mask=dist_resized)
            image_tensor = transformed['image']
            dist_tensor = transformed['mask']

            # Ensure float32 tensor
            if isinstance(dist_tensor, np.ndarray):
                dist_tensor = torch.from_numpy(dist_tensor).float()
            else:
                dist_tensor = dist_tensor.float()

        except Exception as e:
            # Fallback: simple resize and normalize
            print(f"Augmentation failed: {e}, using fallback")
            image_resized = cv2.resize(image, (image_size, image_size))
            dist_resized = cv2.resize(distance_field, (image_size, image_size)).astype(np.float32)

            normalize = A.Compose([
                A.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
            image_tensor = normalize(image=image_resized)['image']
            dist_tensor = torch.from_numpy(dist_resized).float()

        # Downsample distance field to match model output
        output_size = image_size // 4
        if dist_tensor.dim() == 2:
            dist_tensor = dist_tensor.unsqueeze(0)

        # Ensure float32 before interpolation
        dist_tensor = dist_tensor.float()

        dist_downsampled = F.interpolate(
            dist_tensor.unsqueeze(0),
            size=(output_size, output_size),
            mode='bilinear',
            align_corners=False
        ).squeeze(0)

        # Final check: ensure both are float32
        image_tensor = image_tensor.float()
        dist_downsampled = dist_downsampled.float()

        return image_tensor, dist_downsampled



# MODAL

In [ ]:
# @title
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from typing import List, Tuple, Optional

class ModelConfig:
    """Cấu hình tập trung các hyperparameters quan trọng"""
    def __init__(self):
        # ===== BACKBONE =====
        self.backbone = "tf_efficientnetv2_s"
        self.pretrained = True

        # ===== BOTTLENECK =====
        self.bottleneck_channels = 256  # Target channels sau encoder

        # ===== ASPP Configuration =====
        self.aspp_dilations = [1, 6, 12, 18]
        self.aspp_dropout = 0.15

        # ===== Pyramid Pooling =====
        self.ppm_scales = [1, 2, 3, 6]

        # ===== Directional Convolution =====
        self.dir_conv_kernel_sizes = {
            'vertical': (7, 1),
            'horizontal': (1, 7),
            'diagonal': 7
        }

        # ===== Spine Shape Attention =====
        self.spine_att_scales = [3, 5, 7]
        self.spine_vertical_kernel = (11, 1)

        # ===== Curvature Detection =====
        self.curvature_kernel_size = 3

        # ===== Edge Detection =====
        self.edge_detect_scales = [3, 5]

        # ===== Decoder =====
        self.decoder_channels = [256, 128, 64, 32]
        self.decoder_kernel_size = 3

        # ===== CBAM Attention =====
        self.cbam_reduction = 16
        self.cbam_spatial_kernel = 7

        # ===== Deep Supervision =====
        self.use_deep_supervision = True
        self.ds_weights = [0.5, 0.3, 0.2]

        # ===== Training =====
        self.dropout_rate = 0.1
        self.use_residual = True

    def print_config(self):
        print("="*70)
        print(" MODEL CONFIGURATION ".center(70, "="))
        print("="*70)
        for key, value in self.__dict__.items():
            print(f"  {key:30s}: {value}")
        print("="*70)


class ChannelAdapter(nn.Module):
    """Adapter để chuyển đổi channels linh hoạt"""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        if in_channels != out_channels:
            self.adapt = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True)
            )
        else:
            self.adapt = nn.Identity()

    def forward(self, x):
        return self.adapt(x)


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling - Fixed channels"""
    def __init__(self, channels: int, dilations: List[int], dropout: float = 0.15):
        super().__init__()
        branch_ch = channels // 2

        def conv_1x1(ch):
            return nn.Sequential(
                nn.Conv2d(ch, branch_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            )

        def atrous_conv(ch, dilation):
            return nn.Sequential(
                nn.Conv2d(ch, branch_ch, kernel_size=3, padding=dilation,
                         dilation=dilation, bias=False),
                nn.BatchNorm2d(branch_ch),
                nn.ReLU(inplace=True),
            )

        self.branches = nn.ModuleList()
        for dil in dilations:
            if dil == 1:
                self.branches.append(conv_1x1(channels))
            else:
                self.branches.append(atrous_conv(channels, dil))

        self.global_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, branch_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(branch_ch),
            nn.ReLU(inplace=True),
        )

        total_ch = branch_ch * (len(dilations) + 1)
        self.conv_out = nn.Sequential(
            nn.Conv2d(total_ch, channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        feats = [branch(x) for branch in self.branches]
        global_feat = F.interpolate(self.global_pool(x), size=x.shape[2:],
                                     mode='bilinear', align_corners=False)
        feats.append(global_feat)
        return self.conv_out(torch.cat(feats, dim=1))


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_map, max_map], dim=1)
        att = self.sigmoid(self.conv(x_cat))
        return x * att


class CBAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 16, spatial_kernel: int = 7):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, max(channels // reduction, 8), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // reduction, 8), channels, 1, bias=False)
        )

        self.sigmoid_c = nn.Sigmoid()
        self.spatial = SpatialAttention(kernel_size=spatial_kernel)

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        channel_att = self.sigmoid_c(avg_out + max_out)
        x = x * channel_att
        return self.spatial(x)


class DirectionalConv(nn.Module):
    def __init__(self, channels: int, kernel_config: dict):
        super().__init__()

        v_kernel = kernel_config['vertical']
        h_kernel = kernel_config['horizontal']
        d_kernel = kernel_config['diagonal']

        self.vertical = nn.Conv2d(channels, channels, kernel_size=v_kernel,
                                  padding=(v_kernel[0]//2, v_kernel[1]//2),
                                  groups=channels, bias=False)

        self.horizontal = nn.Conv2d(channels, channels, kernel_size=h_kernel,
                                     padding=(h_kernel[0]//2, h_kernel[1]//2),
                                     groups=channels, bias=False)

        self.diag1 = nn.Conv2d(channels, channels, kernel_size=d_kernel,
                               padding=d_kernel//2, groups=channels, bias=False)
        self.diag2 = nn.Conv2d(channels, channels, kernel_size=d_kernel,
                               padding=d_kernel//2, groups=channels, bias=False)

        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 4, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        v = self.vertical(x)
        h = self.horizontal(x)
        d1 = self.diag1(x)
        d2 = self.diag2(x)
        out = self.fusion(torch.cat([v, h, d1, d2], dim=1))
        return out + x


class SpineShapeAttention(nn.Module):
    def __init__(self, channels: int, scales: List[int], vertical_kernel: Tuple[int, int]):
        super().__init__()
        num_scales = len(scales)
        scale_ch = max(channels // num_scales, 16)

        self.shape_convs = nn.ModuleList()
        for scale in scales:
            padding = scale // 2
            self.shape_convs.append(
                nn.Conv2d(channels, scale_ch, kernel_size=scale, padding=padding)
            )

        self.vertical_consistency = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=vertical_kernel,
                     padding=(vertical_kernel[0]//2, vertical_kernel[1]//2)),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

        self.smoothness = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=5, padding=2, groups=channels),
            nn.BatchNorm2d(channels),
            nn.Sigmoid()
        )

        total_ch = channels * 2 + scale_ch * num_scales
        self.attention = nn.Sequential(
            nn.Conv2d(total_ch, channels, 1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        shape_feats = [conv(x) for conv in self.shape_convs]
        vc = self.vertical_consistency(x)
        smooth = self.smoothness(x)
        x_smooth = x * smooth
        combined = torch.cat([x, vc] + shape_feats, dim=1)
        att = self.attention(combined)
        return x_smooth * att


class CurvatureDetection(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()
        padding = kernel_size // 2

        self.dx_conv = nn.Conv2d(channels, channels, kernel_size, padding=padding,
                                groups=channels, bias=False)
        self.dy_conv = nn.Conv2d(channels, channels, kernel_size, padding=padding,
                                groups=channels, bias=False)

        self.dxx_conv = nn.Conv2d(channels, channels, kernel_size, padding=padding,
                                 groups=channels, bias=False)
        self.dyy_conv = nn.Conv2d(channels, channels, kernel_size, padding=padding,
                                 groups=channels, bias=False)
        self.dxy_conv = nn.Conv2d(channels, channels, kernel_size, padding=padding,
                                 groups=channels, bias=False)

        self.curvature_fusion = nn.Sequential(
            nn.Conv2d(channels * 5, channels, 1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        dx = self.dx_conv(x)
        dy = self.dy_conv(x)
        dxx = self.dxx_conv(dx)
        dyy = self.dyy_conv(dy)
        dxy = self.dxy_conv(dx)
        curvature_features = torch.cat([dx, dy, dxx, dyy, dxy], dim=1)
        return self.curvature_fusion(curvature_features)


class EdgeAttention(nn.Module):
    def __init__(self, channels: int, scales: List[int] = [3, 5]):
        super().__init__()
        scale_ch = max(channels // len(scales), 16)

        self.edge_convs = nn.ModuleList()
        for scale in scales:
            padding = scale // 2
            self.edge_convs.append(
                nn.Sequential(
                    nn.Conv2d(channels, scale_ch, scale, padding=padding),
                    nn.BatchNorm2d(scale_ch),
                    nn.ReLU(inplace=True)
                )
            )

        self.edge_out = nn.Sequential(
            nn.Conv2d(channels + scale_ch * len(scales), channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        edge_feats = [conv(x) for conv in self.edge_convs]
        edges = torch.cat([x] + edge_feats, dim=1)
        edge_map = self.edge_out(edges)
        return x * edge_map


class PyramidPooling(nn.Module):
    def __init__(self, channels: int, scales: List[int]):
        super().__init__()
        pool_ch = max(channels // len(scales), 16)

        self.pools = nn.ModuleList()
        self.convs = nn.ModuleList()

        for scale in scales:
            self.pools.append(nn.AdaptiveAvgPool2d(scale))
            self.convs.append(
                nn.Sequential(
                    nn.Conv2d(channels, pool_ch, 1, bias=False),
                    nn.BatchNorm2d(pool_ch),
                    nn.ReLU(inplace=True)
                )
            )

        total_ch = channels + pool_ch * len(scales)
        self.out_conv = nn.Sequential(
            nn.Conv2d(total_ch, channels, 1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        size = x.shape[2:]
        feats = [x]

        for pool, conv in zip(self.pools, self.convs):
            pooled = pool(x)
            feat = conv(pooled)
            upsampled = F.interpolate(feat, size=size, mode='bilinear', align_corners=False)
            feats.append(upsampled)

        return self.out_conv(torch.cat(feats, dim=1))


class SpineCurveDetectorV2(nn.Module):
    """Enhanced Spine Curve Detector V2 - Fixed Channel Architecture"""
    def __init__(self, config: Optional[ModelConfig] = None):
        super().__init__()

        self.config = config if config is not None else ModelConfig()
        self.config.print_config()

        # Encoder
        self.encoder = timm.create_model(
            self.config.backbone,
            pretrained=self.config.pretrained,
            features_only=True,
            out_indices=[1, 2, 3, 4]
        )

        # Get encoder channel dimensions
        enc_ch1 = self.encoder.feature_info[1]["num_chs"]
        enc_ch2 = self.encoder.feature_info[2]["num_chs"]
        enc_ch3 = self.encoder.feature_info[3]["num_chs"]
        enc_ch4 = self.encoder.feature_info[4]["num_chs"]

        print(f"\n{'='*70}")
        print(" ENCODER ARCHITECTURE ".center(70, "="))
        print(f"{'='*70}")
        print(f"  Level 1: {enc_ch1:4d} channels")
        print(f"  Level 2: {enc_ch2:4d} channels")
        print(f"  Level 3: {enc_ch3:4d} channels")
        print(f"  Level 4: {enc_ch4:4d} channels (bottleneck input)")
        print(f"{'='*70}\n")

        # Target channels
        bottleneck_ch = self.config.bottleneck_channels
        dec_ch = self.config.decoder_channels

        # CRITICAL: Adapt ALL encoder outputs to expected channels
        self.enc1_adapter = ChannelAdapter(enc_ch1, dec_ch[3])  # For dec1 skip
        self.enc2_adapter = ChannelAdapter(enc_ch2, dec_ch[2])  # For dec2 skip
        self.enc3_adapter = ChannelAdapter(enc_ch3, dec_ch[1])  # For dec3 skip
        self.enc4_adapter = ChannelAdapter(enc_ch4, bottleneck_ch)  # For bottleneck

        print(f"Channel Adapters:")
        print(f"  e1: {enc_ch1:4d} -> {dec_ch[3]:4d}")
        print(f"  e2: {enc_ch2:4d} -> {dec_ch[2]:4d}")
        print(f"  e3: {enc_ch3:4d} -> {dec_ch[1]:4d}")
        print(f"  e4: {enc_ch4:4d} -> {bottleneck_ch:4d}")
        print(f"{'='*70}\n")

        # Bottleneck modules (all work with bottleneck_ch)
        self.aspp = ASPP(bottleneck_ch, self.config.aspp_dilations, self.config.aspp_dropout)
        self.ppm = PyramidPooling(bottleneck_ch, self.config.ppm_scales)

        # Decoder blocks
        self.dec4 = self._make_block(bottleneck_ch, dec_ch[0])
        self.dec3 = self._make_block(dec_ch[0] + dec_ch[1], dec_ch[1])
        self.dec2 = self._make_block(dec_ch[1] + dec_ch[2], dec_ch[2])
        self.dec1 = self._make_block(dec_ch[2] + dec_ch[3], dec_ch[3])

        # Enhancement modules
        self.dir_conv4 = DirectionalConv(dec_ch[0], self.config.dir_conv_kernel_sizes)
        self.dir_conv3 = DirectionalConv(dec_ch[1], self.config.dir_conv_kernel_sizes)
        self.dir_conv2 = DirectionalConv(dec_ch[2], self.config.dir_conv_kernel_sizes)
        self.dir_conv1 = DirectionalConv(dec_ch[3], self.config.dir_conv_kernel_sizes)

        self.spine_att4 = SpineShapeAttention(dec_ch[0], self.config.spine_att_scales, self.config.spine_vertical_kernel)
        self.spine_att3 = SpineShapeAttention(dec_ch[1], self.config.spine_att_scales, self.config.spine_vertical_kernel)
        self.spine_att2 = SpineShapeAttention(dec_ch[2], self.config.spine_att_scales, self.config.spine_vertical_kernel)

        self.curve_det4 = CurvatureDetection(dec_ch[0], self.config.curvature_kernel_size)
        self.curve_det3 = CurvatureDetection(dec_ch[1], self.config.curvature_kernel_size)
        self.curve_det2 = CurvatureDetection(dec_ch[2], self.config.curvature_kernel_size)

        self.edge_att4 = EdgeAttention(dec_ch[0], self.config.edge_detect_scales)
        self.edge_att3 = EdgeAttention(dec_ch[1], self.config.edge_detect_scales)
        self.edge_att2 = EdgeAttention(dec_ch[2], self.config.edge_detect_scales)

        # Skip refinement
        self.refine3 = nn.Sequential(
            CBAM(dec_ch[1], self.config.cbam_reduction, self.config.cbam_spatial_kernel)
        )
        self.refine2 = nn.Sequential(
            CBAM(dec_ch[2], self.config.cbam_reduction, self.config.cbam_spatial_kernel)
        )
        self.refine1 = nn.Sequential(
            CBAM(dec_ch[3], self.config.cbam_reduction, self.config.cbam_spatial_kernel)
        )

        # Final prediction
        self.final = nn.Sequential(
            nn.Conv2d(dec_ch[3], dec_ch[3], 3, padding=1),
            nn.BatchNorm2d(dec_ch[3]),
            nn.ReLU(inplace=True),
            nn.Dropout(self.config.dropout_rate),
            nn.Conv2d(dec_ch[3], 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 1)
        )

        # Deep supervision
        if self.config.use_deep_supervision:
            self.ds_conv2 = nn.Conv2d(dec_ch[2], 1, 1)
            self.ds_conv3 = nn.Conv2d(dec_ch[1], 1, 1)
            self.ds_conv4 = nn.Conv2d(dec_ch[0], 1, 1)

    def _make_block(self, in_ch: int, out_ch: int):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, self.config.decoder_kernel_size,
                     padding=self.config.decoder_kernel_size//2),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            CBAM(out_ch, self.config.cbam_reduction, self.config.cbam_spatial_kernel),
            nn.Conv2d(out_ch, out_ch, self.config.decoder_kernel_size,
                     padding=self.config.decoder_kernel_size//2),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        # Encoder với channel adaptation
        e1, e2, e3, e4 = self.encoder(x)

        e1 = self.enc1_adapter(e1)
        e2 = self.enc2_adapter(e2)
        e3 = self.enc3_adapter(e3)
        e4 = self.enc4_adapter(e4)

        # Bottleneck
        e4 = self.aspp(e4)
        e4 = self.ppm(e4)

        # Decoder 4
        d4 = self.dec4(e4)
        d4 = self.dir_conv4(d4)
        d4 = self.spine_att4(d4)
        d4_curve = self.curve_det4(d4)
        d4 = d4 + d4_curve if self.config.use_residual else d4_curve
        d4 = self.edge_att4(d4)
        d4_up = F.interpolate(d4, size=e3.shape[2:], mode="bilinear", align_corners=False)

        # Decoder 3
        e3 = self.refine3(e3)
        d3 = self.dec3(torch.cat([d4_up, e3], dim=1))
        d3 = self.dir_conv3(d3)
        d3 = self.spine_att3(d3)
        d3_curve = self.curve_det3(d3)
        d3 = d3 + d3_curve if self.config.use_residual else d3_curve
        d3 = self.edge_att3(d3)
        d3_up = F.interpolate(d3, size=e2.shape[2:], mode="bilinear", align_corners=False)

        # Decoder 2
        e2 = self.refine2(e2)
        d2 = self.dec2(torch.cat([d3_up, e2], dim=1))
        d2 = self.dir_conv2(d2)
        d2 = self.spine_att2(d2)
        d2_curve = self.curve_det2(d2)
        d2 = d2 + d2_curve if self.config.use_residual else d2_curve
        d2 = self.edge_att2(d2)
        d2_up = F.interpolate(d2, size=e1.shape[2:], mode="bilinear", align_corners=False)

        # Decoder 1
        e1 = self.refine1(e1)
        d1 = self.dec1(torch.cat([d2_up, e1], dim=1))
        d1 = self.dir_conv1(d1)

        # Final
        out = self.final(d1)

        # Deep supervision
        if self.config.use_deep_supervision:
            ds4 = F.interpolate(self.ds_conv4(d4), size=out.shape[2:],
                               mode="bilinear", align_corners=False)
            ds3 = F.interpolate(self.ds_conv3(d3), size=out.shape[2:],
                               mode="bilinear", align_corners=False)
            ds2 = F.interpolate(self.ds_conv2(d2), size=out.shape[2:],
                               mode="bilinear", align_corners=False)
            return out, [ds2, ds3, ds4]

        return out, []

    def get_num_params(self):
        return sum(p.numel() for p in self.parameters())

    def get_model_size(self):
        param_size = sum(p.numel() * p.element_size() for p in self.parameters())
        buffer_size = sum(b.numel() * b.element_size() for b in self.buffers())
        return (param_size + buffer_size) / 1024 / 1024


Spine

In [ ]:
# @title
from typing import List, Tuple, Optional
import math


class SpineConfig:
    """Cấu hình cho việc sinh spine"""
    def __init__(self):
        # Độ mạnh của lực hút về target points (0.0 - 1.0)
        self.target_pull_strength = 0.35

        # Số lần lặp FABRIK
        self.fabrik_iterations = 60

        # Số lần lặp smooth cuối cùng
        self.final_smooth_iterations = 3

        # Hệ số làm mượt trong quá trình FABRIK (0.0 - 1.0)
        # Càng cao càng mượt nhưng mất chi tiết
        self.smoothing_factor = 0.15

        # Ngưỡng phát hiện đường cong (% của khoảng cách đầu-cuối)
        self.curve_detection_threshold = 0.03

        # Cho phép spine vượt quá target points để tạo độ cong mạnh hơn
        self.allow_overshoot = True

        # Hệ số overshoot (1.0 = không overshoot, >1.0 = overshoot mạnh hơn)
        self.overshoot_factor = 1.2

        # Sử dụng local bending force cho từng segment
        self.use_local_bending = True

        # Độ mạnh của local bending (0.0 - 1.0)
        self.local_bending_strength = 0.25

        # Bán kính ảnh hưởng của local bending (số segments)
        self.local_bending_radius = 3

        # Sử dụng segment length flexibility (cho phép segments dài/ngắn khác nhau)
        self.use_flexible_segments = True

        # Độ linh hoạt của segment length (1.0 = cố định, >1.0 = linh hoạt hơn)
        self.segment_flexibility = 1.3

        # Áp dụng multi-pass bending (nhiều lượt uốn với độ mạnh tăng dần)
        self.use_multi_pass = True

        # Số lượt multi-pass
        self.multi_pass_count = 3

        # Sử dụng adaptive smoothing (smooth ít hơn ở vùng cong nhiều)
        self.use_adaptive_smoothing = True


class SpineGenerator:
    def __init__(self, joint_count: int = 18, max_angle: float = 45.0, config: Optional[SpineConfig] = None):
        """
        Khởi tạo SpineGenerator

        Args:
            joint_count: Số lượng khớp (mặc định 18)
            max_angle: Góc tối đa giữa 2 đốt sống (độ)
            config: Cấu hình tùy chỉnh (nếu None sẽ dùng mặc định)
        """
        self.joint_count = joint_count
        self.max_angle = max_angle
        self.max_angle_rad = math.radians(max_angle)
        self.config = config if config is not None else SpineConfig()

    def distance_point_to_line(
        self,
        point: Tuple[float, float],
        start: Tuple[float, float],
        end: Tuple[float, float]
    ) -> float:
        """Tính khoảng cách có dấu từ point đến đường thẳng start-end"""
        px, py = point
        sx, sy = start
        ex, ey = end

        dx = ex - sx
        dy = ey - sy
        L = math.hypot(dx, dy)

        if L < 1e-6:
            return math.hypot(px - sx, py - sy)

        perp_x = -dy / L
        perp_y = dx / L

        t = ((px - sx) * dx + (py - sy) * dy) / (L * L)
        proj_x = sx + t * dx
        proj_y = sy + t * dy

        return (px - proj_x) * perp_x + (py - proj_y) * perp_y

    def create_straight_spine(self, start: Tuple[float, float],
                             end: Tuple[float, float]) -> List[Tuple[float, float]]:
        """Tạo spine thẳng từ start đến end"""
        joints = []
        for i in range(self.joint_count):
            t = i / (self.joint_count - 1)
            x = start[0] + (end[0] - start[0]) * t
            y = start[1] + (end[1] - start[1]) * t
            joints.append((x, y))
        return joints

    def interpolate_targets(
        self,
        target_points: List[Tuple[float, float]],
        joint_count: int
    ) -> List[Tuple[float, float]]:
        """Nội suy các target points thành joint_count điểm"""
        t_src = np.linspace(0, 1, len(target_points))
        t_dst = np.linspace(0, 1, joint_count)

        xs = np.interp(t_dst, t_src, [p[0] for p in target_points])
        ys = np.interp(t_dst, t_src, [p[1] for p in target_points])

        return list(zip(xs, ys))

    def apply_local_bending(
        self,
        spine: List[List[float]],
        joint_targets: List[Tuple[float, float]],
        fixed_joints: dict
    ) -> None:
        """
        Áp dụng lực uốn cục bộ để spine đi qua các target points tốt hơn
        """
        if not self.config.use_local_bending:
            return

        radius = self.config.local_bending_radius
        strength = self.config.local_bending_strength

        for i in range(len(spine)):
            if i in fixed_joints:
                continue

            # Tính vector từ spine hiện tại đến target
            tx, ty = joint_targets[i]
            dx = tx - spine[i][0]
            dy = ty - spine[i][1]
            dist = math.hypot(dx, dy)

            if dist < 1e-6:
                continue

            # Ảnh hưởng lên các joints lân cận
            for j in range(max(0, i - radius), min(len(spine), i + radius + 1)):
                if j in fixed_joints:
                    continue

                # Tính độ ảnh hưởng theo khoảng cách
                offset = abs(j - i)
                influence = (1.0 - offset / (radius + 1)) * strength

                # Áp dụng lực
                spine[j][0] += dx * influence
                spine[j][1] += dy * influence

    def smooth_spine(
        self,
        joints: List[Tuple[float, float]],
        start: Tuple[float, float],
        end: Tuple[float, float],
        iterations: int = 15
    ) -> List[Tuple[float, float]]:

        """Làm mượt spine với ràng buộc start và end"""
        new_joints = [list(j) for j in joints]
        total_length = math.dist(start, end)
        seg_len = total_length / (self.joint_count - 1)

        for _ in range(iterations):
            # Forward pass
            new_joints[0] = list(start)
            for i in range(1, self.joint_count):
                dx = new_joints[i][0] - new_joints[i - 1][0]
                dy = new_joints[i][1] - new_joints[i - 1][1]
                d = math.hypot(dx, dy) + 1e-6
                new_joints[i][0] = new_joints[i - 1][0] + dx / d * seg_len
                new_joints[i][1] = new_joints[i - 1][1] + dy / d * seg_len

            # Backward pass
            new_joints[-1] = list(end)
            for i in range(self.joint_count - 2, -1, -1):
                dx = new_joints[i][0] - new_joints[i + 1][0]
                dy = new_joints[i][1] - new_joints[i + 1][1]
                d = math.hypot(dx, dy) + 1e-6
                new_joints[i][0] = new_joints[i + 1][0] + dx / d * seg_len
                new_joints[i][1] = new_joints[i + 1][1] + dy / d * seg_len

        return [tuple(j) for j in new_joints]

    def bend_spine_through_points(self, spine, target_points):
        import math
        import numpy as np

        joint_count = len(spine)
        start = spine[0]
        end = spine[-1]

        new_spine = [list(p) for p in spine]

        # ==================================================
        # 1. TRỤC + PHÁP TUYẾN
        # ==================================================
        axis = np.array(end, dtype=np.float64) - np.array(start, dtype=np.float64)
        axis /= (np.linalg.norm(axis) + 1e-6)
        normal = np.array([-axis[1], axis[0]], dtype=np.float64)

        # ==================================================
        # 2. SIGNED DISTANCE → PHÂN LOẠI CONG
        # ==================================================
        distances = []
        for p in target_points:
            v = np.array(p) - np.array(start)
            proj = np.dot(v, axis) * axis
            signed = np.dot(v - proj, normal)
            distances.append(signed)

        max_d = max(distances)
        min_d = min(distances)

        H = math.dist(start, end)
        thresh = 0.01 * H

        if max_d > thresh and min_d < -thresh:
            curve_type = "S"
        elif abs(max_d) > thresh or abs(min_d) > thresh:
            curve_type = "C"
        else:
            curve_type = "STRAIGHT"

        # ==================================================
        # 3. TRƯỜNG HỢP THẲNG
        # ==================================================
        if curve_type == "STRAIGHT":
            return [
                (
                    start[0] + i * (end[0] - start[0]) / (joint_count - 1),
                    start[1] + i * (end[1] - start[1]) / (joint_count - 1)
                )
                for i in range(joint_count)
            ]

        # ==================================================
        # 4. ĐỘ DÀI KHỚP
        # ==================================================
        avg_len = H / (joint_count - 1)

        # ==================================================
        # 5. FIXED JOINTS (INFLECTION CHO S)
        # ==================================================
        fixed_joints = {
            0: start,
            joint_count - 1: end
        }

        # if curve_type == "S":
        #     i_pos = distances.index(max_d)
        #     i_neg = distances.index(min_d)
        #     a, b = sorted([i_pos, i_neg])

        #     if b - a > 1:
        #         mid_idx = min(range(a + 1, b), key=lambda i: abs(distances[i]))
        #         t = mid_idx / (len(distances) - 1)
        #         fixed_joints[joint_count // 2] = (
        #             start[0] + t * (end[0] - start[0]),
        #             start[1] + t * (end[1] - start[1])
        #         )

        # ==================================================
        # 6. TARGET NỘI SUY
        # ==================================================
        joint_targets = self.interpolate_targets(target_points, joint_count)

        # ==================================================
        # 7. GAUSSIAN
        # ==================================================
        sigma = 0.12 * joint_count

        def gaussian(i, c):
            return math.exp(-((i - c) ** 2) / (2 * sigma ** 2))

        if curve_type == "S":
            center_pos = distances.index(max_d) / (len(distances) - 1) * (joint_count - 1)
            center_neg = distances.index(min_d) / (len(distances) - 1) * (joint_count - 1)
        else:
            center_pos = joint_count // 2

        # ==================================================
        # 8. FABRIK + LỰC 2D
        # ==================================================
        for it in range(self.config.fabrik_iterations):
            strength = self.config.target_pull_strength * (1.0 + 1.2 * it / self.config.fabrik_iterations)

            # ---------- FORWARD ----------
            for i in range(1, joint_count):
                if i in fixed_joints:
                    new_spine[i] = list(fixed_joints[i])
                    continue

                # --- length constraint ---
                dx = new_spine[i][0] - new_spine[i - 1][0]
                dy = new_spine[i][1] - new_spine[i - 1][1]
                d = math.hypot(dx, dy) + 1e-6

                if curve_type == "S":
                    w_len = max(gaussian(i, center_pos), gaussian(i, center_neg))
                else:
                    w_len = gaussian(i, center_pos)

                min_len = avg_len * (0.70 - 0.25 * w_len)
                max_len = avg_len * (1.30 + 0.50 * w_len)

                d_target = max(min_len, min(d, max_len))
                scale = d_target / d

                new_spine[i][0] = new_spine[i - 1][0] + dx * scale
                new_spine[i][1] = new_spine[i - 1][1] + dy * scale

                # ---------- LỰC 2D ----------
                p = np.array(new_spine[i], dtype=np.float64)
                tgt = np.array(joint_targets[i], dtype=np.float64)

                vec = tgt - p

                f_para = np.dot(vec, axis) * axis
                f_perp = vec - f_para

                if curve_type == "S":
                    w = gaussian(i, center_pos) + gaussian(i, center_neg)
                else:
                    w = gaussian(i, center_pos)

                force = (
                    0.9 * f_perp * w +
                    0.3 * f_para
                )

                # clamp lực
                max_force = 0.08 * H
                norm_f = np.linalg.norm(force)
                if norm_f > max_force:
                    force *= max_force / (norm_f + 1e-6)

                new_spine[i][0] += force[0] * strength
                new_spine[i][1] += force[1] * strength

            # ---------- BACKWARD ----------
            for i in range(joint_count - 2, -1, -1):
                if i in fixed_joints:
                    new_spine[i] = list(fixed_joints[i])
                    continue

                dx = new_spine[i][0] - new_spine[i + 1][0]
                dy = new_spine[i][1] - new_spine[i + 1][1]
                d = math.hypot(dx, dy) + 1e-6

                d_target = max(avg_len * 0.7, min(d, avg_len * 1.3))
                scale = d_target / d

                new_spine[i][0] = new_spine[i + 1][0] + dx * scale
                new_spine[i][1] = new_spine[i + 1][1] + dy * scale

        return [tuple(p) for p in new_spine]


    def generate_spine(self, points: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
        """
        Hàm chính: Tạo spine từ list điểm

        Args:
            points: List điểm đầu vào [(x, y), ...]

        Returns:
            List khớp spine [(x, y), ...]
        """
        if len(points) < 2:
            raise ValueError("Cần ít nhất 2 điểm")

        start = points[0]
        end = points[-1]

        # Tạo spine thẳng ban đầu
        spine = self.create_straight_spine(start, end)

        # Nếu có điểm giữa, uốn spine
        if len(points) > 2:
            spine = self.bend_spine_through_points(spine, points)

        return spine

    def draw_spine_on_image(self, image: np.ndarray,
                        input_points: List[Tuple[float, float]],
                        spine_joints: List[Tuple[float, float]]) -> np.ndarray:
        """Vẽ spine và input points lên ảnh"""
        result = image.copy()

        # Vẽ đường nối input points (đỏ nét đứt)
        for i in range(len(input_points) - 1):
            pt1 = (int(input_points[i][0]), int(input_points[i][1]))
            pt2 = (int(input_points[i+1][0]), int(input_points[i+1][1]))
            self._draw_dashed_line(result, pt1, pt2, (239, 68, 68), thickness=3)

        # Vẽ input points
        for i, point in enumerate(input_points):
            center = (int(point[0]), int(point[1]))
            radius = 10 if (i == 0 or i == len(input_points) - 1) else 7
            cv2.circle(result, center, radius, (239, 68, 68), -1)
            cv2.circle(result, center, radius, (255, 255, 255), 2)

        # Vẽ spine (xanh dương)
        spine_points = np.array([[int(j[0]), int(j[1])] for j in spine_joints], np.int32)

        for i in range(len(spine_points) - 1):
            cv2.line(result, tuple(spine_points[i]), tuple(spine_points[i+1]),
                    (59, 130, 246), thickness=12, lineType=cv2.LINE_AA)

        # Vẽ joints
        for i, joint in enumerate(spine_joints):
            center = (int(joint[0]), int(joint[1]))
            radius = 12 if (i == 0 or i == len(spine_joints) - 1) else 7
            shadow_center = (center[0] + 2, center[1] + 2)
            cv2.circle(result, shadow_center, radius, (0, 0, 0), -1)
            cv2.circle(result, center, radius, (30, 64, 175), -1)
            cv2.circle(result, center, radius, (96, 165, 250), 2)

        return result

    def _draw_dashed_line(self, image: np.ndarray, pt1: Tuple[int, int],
                        pt2: Tuple[int, int], color: Tuple[int, int, int],
                        thickness: int = 2, dash_length: int = 10):
        """Vẽ đường nét đứt"""
        dist = math.sqrt((pt2[0] - pt1[0]) ** 2 + (pt2[1] - pt1[1]) ** 2)
        dashes = int(dist / dash_length)

        for i in range(dashes):
            if i % 2 == 0:
                start = (int(pt1[0] + (pt2[0] - pt1[0]) * i / dashes),
                        int(pt1[1] + (pt2[1] - pt1[1]) * i / dashes))
                end = (int(pt1[0] + (pt2[0] - pt1[0]) * (i + 1) / dashes),
                    int(pt1[1] + (pt2[1] - pt1[1]) * (i + 1) / dashes))
                cv2.line(image, start, end, color, thickness, lineType=cv2.LINE_AA)


# ============================================================================
# EXAMPLE USAGE
# ============================================================================

def spine_visual(input_points, image):

    # Config 2: Aggressive (cho cong nặng)
    config_aggressive = SpineConfig()
    config_aggressive.target_pull_strength = 0.8
    config_aggressive.fabrik_iterations = 150
    config_aggressive.overshoot_factor = 1.2
    config_aggressive.local_bending_strength = 0.35
    config_aggressive.final_smooth_iterations = 2


    generator_aggressive = SpineGenerator(joint_count=18, max_angle=45.0, config=config_aggressive)
    spine_aggressive = generator_aggressive.generate_spine(input_points)
    result_aggressive = generator_aggressive.draw_spine_on_image(image.copy(), input_points, spine_aggressive)
    cv2.imwrite('spine_aggressive.png', cv2.cvtColor(result_aggressive, cv2.COLOR_RGB2BGR))

    return result_aggressive, spine_aggressive


In [ ]:
# @title

# ============================================================================
# CURVE GENERATION UTILITIES
# ============================================================================

class SpineCurveGenerator:
    """Generate smooth spine curve from landmarks"""

    @staticmethod
    def fit_smooth_curve(landmarks, num_points=200, smoothing=0.5):
        """
        Fit smooth B-spline curve through landmarks
        Args:
            landmarks: (N, 2) array of (x, y) coordinates
            num_points: number of points on final curve
            smoothing: smoothing parameter (0 = interpolate, higher = smoother)
        Returns:
            curve_points: (num_points, 2) smooth curve coordinates
        """
        # Remove NaN values
        valid_mask = ~np.isnan(landmarks).any(axis=1)
        valid_landmarks = landmarks[valid_mask]

        if len(valid_landmarks) < 3:
            return None

        # Sort by y-coordinate (top to bottom)
        sorted_idx = np.argsort(valid_landmarks[:, 1])
        sorted_landmarks = valid_landmarks[sorted_idx]

        try:
            # Fit parametric B-spline
            tck, u = splprep([sorted_landmarks[:, 0], sorted_landmarks[:, 1]],
                            s=smoothing * len(sorted_landmarks), k=min(3, len(sorted_landmarks)-1))

            # Evaluate spline at uniform intervals
            u_new = np.linspace(0, 1, num_points)
            x_new, y_new = splev(u_new, tck)

            curve_points = np.column_stack([x_new, y_new])
            return curve_points

        except:
            # Fallback: simple interpolation
            if len(sorted_landmarks) > 1:
                from scipy.interpolate import interp1d
                t = np.linspace(0, 1, len(sorted_landmarks))
                t_new = np.linspace(0, 1, num_points)

                fx = interp1d(t, sorted_landmarks[:, 0], kind='cubic', fill_value='extrapolate')
                fy = interp1d(t, sorted_landmarks[:, 1], kind='cubic', fill_value='extrapolate')

                return np.column_stack([fx(t_new), fy(t_new)])
            return None

    @staticmethod
    def generate_curve_mask(curve_points, image_shape, thickness=8):
        """
        Generate binary mask from curve
        Args:
            curve_points: (N, 2) curve coordinates
            image_shape: (H, W) output shape
            thickness: line thickness in pixels
        Returns:
            mask: (H, W) binary mask
        """
        mask = np.zeros(image_shape, dtype=np.uint8)

        if curve_points is None or len(curve_points) < 2:
            return mask

        # Draw curve on mask
        pts = curve_points.astype(np.int32)
        cv2.polylines(mask, [pts], False, 255, thickness=thickness, lineType=cv2.LINE_AA)

        return mask.astype(np.float32) / 255.0

    @staticmethod
    def generate_distance_field(curve_points, image_shape, max_distance=50):
        """
        Generate distance field (useful for regression)
        Args:
            curve_points: (N, 2) curve coordinates
            image_shape: (H, W) output shape
            max_distance: maximum distance value
        Returns:
            distance_field: (H, W) normalized distance field
        """
        mask = SpineCurveGenerator.generate_curve_mask(curve_points, image_shape, thickness=2)

        # Compute distance transform
        binary_mask = (mask > 0).astype(np.uint8)
        distance = distance_transform_edt(1 - binary_mask)

        # Normalize
        distance = np.clip(distance, 0, max_distance)
        distance_field = 1.0 - (distance / max_distance)

        return distance_field

    @staticmethod
    def visualize_curve(image, landmarks, curve_points, save_path=None):
        """Visualize landmarks and fitted curve"""
        img_vis = image.copy()

        # Draw curve
        if curve_points is not None:
            pts = curve_points.astype(np.int32)
            cv2.polylines(img_vis, [pts], False, (0, 255, 255), 3, cv2.LINE_AA)

        # Draw landmarks
        for i, (x, y) in enumerate(landmarks):
            if not np.isnan(x):
                cv2.circle(img_vis, (int(x), int(y)), 6, (0, 255, 0), -1)
                cv2.circle(img_vis, (int(x), int(y)), 8, (255, 255, 255), 2)

        if save_path:
            cv2.imwrite(save_path, cv2.cvtColor(img_vis, cv2.COLOR_RGB2BGR))

        return img_vis



# ============================================================================
# INFERENCE
# ============================================================================
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import splprep, splev
from scipy.signal import savgol_filter
from scipy.ndimage import convolve
from skimage.morphology import skeletonize
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import deque
import os
import string
import random


class SpineCurvePredictor:
    """Predict spine curve from image"""

    def __init__(self, model_path: str, device='cuda', threshold=0.5):
        """
        Initialize spine curve predictor

        Args:
            model_path: Path to model checkpoint
            device: 'cuda' or 'cpu'
            threshold: Binary threshold for mask
        """
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.threshold = threshold
        self.all_angles = []  # Store angles for spine analysis

        # Load model
        # checkpoint = torch.load(model_path, map_location=self.device)
        # self.model = SpineCurveDetector(backbone="tf_efficientnetv2_m")
        # self.model.load_state_dict(checkpoint['model_state_dict'])
        # self.model = self.model.to(self.device)
        # self.model.eval()

        checkpoint = torch.load(model_path, map_location=self.device)

        config = ModelConfig()
        config.spine_vertical_kernel = (13, 1)
        config.spine_att_scales = [3, 5, 7, 9]

        self.model = SpineCurveDetectorV2(config)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model = self.model.to(self.device)
        self.model.eval()

        print(f"✓ Model loaded from {model_path}")
        print(f"  Best Val Loss: {checkpoint['val_loss']:.4f}")

    # ==================== MASK PROCESSING ====================

    def cover_pred_to_binary(self, mask_pred):
        """
        Convert prediction mask to binary mask with morphological operations

        Args:
            mask_pred: Predicted mask (float values 0-1)

        Returns:
            Binary mask (0 or 255)
        """
        # Threshold mask
        binary_mask = (mask_pred > self.threshold).astype(np.uint8) * 255

        # Morphological operations to clean up
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel)

        return binary_mask

    def _skeletonize(self, binary_mask):
        """
        Skeletonization using morphological thinning

        Args:
            binary_mask: Binary mask

        Returns:
            Skeletonized mask
        """
        from scipy.ndimage import morphology
        skeleton = morphology.binary_erosion(binary_mask, iterations=2)
        return (skeleton * 255).astype(np.uint8)

    def _skeletonize_with_skeletonize(self, binary_mask):
        """
        Skeletonization using scikit-image skeletonize

        Args:
            binary_mask: Binary mask

        Returns:
            Skeletonized mask
        """
        binary = (binary_mask > 0).astype(np.uint8)
        skel = skeletonize(binary)
        return (skel * 255).astype(np.uint8)

    def smooth_from_skeleton(
        self,
        skel,
        radius=10,
        blur_ksize=9,
        blur_sigma=2.0,
        close_ksize=7
    ):
        """
        Advanced smoothing (curvature-preserving):
        - Fill gaps along Y-axis
        - Distance transform fill (soft core)
        - Curvature-aware feathering (KHÔNG mất cực trị)
        - Giữ form cong tự nhiên (cột sống)
        """

        # -----------------------------------------
        # 0) Binary
        # -----------------------------------------
        skel = (skel > 0).astype(np.uint8)

        # -----------------------------------------
        # 1) Vá đứt đoạn theo trục Y
        # -----------------------------------------
        k_close_v = cv2.getStructuringElement(
            cv2.MORPH_RECT, (1, close_ksize)
        )
        skel = cv2.morphologyEx(skel, cv2.MORPH_CLOSE, k_close_v)

        # -----------------------------------------
        # 2) Distance Transform (core fill)
        # -----------------------------------------
        dist = cv2.distanceTransform(skel, cv2.DIST_L2, 5)
        dist = dist / (dist.max() + 1e-6)

        # Mask mềm (chưa threshold gắt)
        soft_mask = np.clip(dist * 255, 0, 255).astype(np.uint8)

        # -----------------------------------------
        # 3) Giãn nhẹ ưu tiên trục Y (không ép ngang)
        # -----------------------------------------
        k_dilate = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (radius // 2 * 2 + 1, radius * 2 + 1)
        )
        soft_mask = cv2.dilate(soft_mask, k_dilate)

        # -----------------------------------------
        # 4) Feather biên CÓ KIỂM SOÁT (2 tầng)
        #    → blur mạnh ở biên, yếu ở lõi
        # -----------------------------------------
        if blur_ksize % 2 == 0:
            blur_ksize += 1

        blur_strong = cv2.GaussianBlur(
            soft_mask, (blur_ksize, blur_ksize), blur_sigma
        )
        blur_weak = cv2.GaussianBlur(
            soft_mask, (3, 3), 0.8
        )

        # -----------------------------------------
        # 5) Trộn theo độ "lõi" (giữ cực trị)
        # -----------------------------------------
        core_weight = np.clip(dist, 0, 1)  # 1 ở lõi, 0 ở biên
        blended = (
            core_weight * blur_weak +
            (1 - core_weight) * blur_strong
        )

        blended = blended.astype(np.uint8)

        # -----------------------------------------
        # 6) Threshold mềm (KHÔNG dùng Otsu)
        # -----------------------------------------
        thresh = int(0.25 * 255)  # mềm hơn Otsu
        mask = (blended > thresh).astype(np.uint8) * 255

        # -----------------------------------------
        # 7) Clean nhẹ (không phá cong)
        # -----------------------------------------
        k_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE, (3, 3)
        )
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k_open)

        k_close_final = cv2.getStructuringElement(
            cv2.MORPH_RECT, (1, 5)
        )
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k_close_final)

        return mask


    # ==================== REGION FILTERING ====================


    def split_mask_by_y_segments(self, binary_mask):
        """
        Chia binary mask thành các segment liên tục theo trục Y
        Mỗi segment là (y_start, y_end)
        """
        H, W = binary_mask.shape
        segments = []

        in_segment = False
        y_start = 0

        for y in range(H):
            row_has_pixel = np.any(binary_mask[y] > 0)

            if row_has_pixel and not in_segment:
                in_segment = True
                y_start = y

            elif not row_has_pixel and in_segment:
                segments.append((y_start, y - 1))
                in_segment = False

        if in_segment:
            segments.append((y_start, H - 1))

        return segments


    def mask_min_distance_dt(self, mask1, mask2):
        """
        Calculate minimum distance between mask1 and mask2 using distance transform

        Args:
            mask1: First binary mask
            mask2: Second binary mask

        Returns:
            Minimum distance value
        """
        inv_mask2 = (mask2 == 0).astype(np.uint8)
        dist_map = cv2.distanceTransform(inv_mask2, cv2.DIST_L2, 5)

        distances = dist_map[mask1 > 0]
        if len(distances) == 0:
            return np.inf

        return distances.min()

    def keep_largest_component_per_y_segment(
        self,
        binary_mask,
        max_y_gap_ratio=0.08
    ):
        """
        Với mỗi segment theo trục Y:
        1. Tìm component có diện tích lớn nhất (main)
        2. Loại component nằm hoàn toàn trong main theo Y
        3. Loại component quá xa main (distance transform)
        """

        H, W = binary_mask.shape
        result = np.zeros_like(binary_mask)

        segments = self.split_mask_by_y_segments(binary_mask)
        max_y_gap = max_y_gap_ratio * H

        for y1, y2 in segments:
            sub_mask = binary_mask[y1:y2 + 1, :]

            num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
                sub_mask, connectivity=8
            )

            if num_labels <= 1:
                continue

            # ==========================
            # 1. MAIN COMPONENT
            # ==========================
            areas = stats[1:, cv2.CC_STAT_AREA]
            main_id = 1 + np.argmax(areas)

            main_stat = stats[main_id]
            main_y1 = main_stat[cv2.CC_STAT_TOP]
            main_y2 = main_y1 + main_stat[cv2.CC_STAT_HEIGHT]

            # mask main (local)
            main_mask = np.zeros_like(sub_mask, dtype=np.uint8)
            main_mask[labels == main_id] = 255

            # ==========================
            # 2. DUYỆT COMPONENT KHÁC
            # ==========================
            for cid in range(1, num_labels):
                stat = stats[cid]
                cy1 = stat[cv2.CC_STAT_TOP]
                cy2 = cy1 + stat[cv2.CC_STAT_HEIGHT]

                # (A) nằm trọn trong main theo Y → loại
                if cy1 > main_y1 and cy2 < main_y2:
                    continue

                # tạo mask cho component hiện tại
                comp_mask = np.zeros_like(sub_mask, dtype=np.uint8)
                comp_mask[labels == cid] = 255

                # (B) quá xa main → loại
                if self.mask_min_distance_dt(comp_mask, main_mask) > max_y_gap:
                    continue

                # (C) GIỮ
                result[y1:y2 + 1][labels == cid] = 255

        return result


    def filter_vertical_regions(
        self,
        binary_mask,
        min_area=60,
        area_weight=1.0,
        height_weight=2.0,
        max_dist_ratio=0.1
    ):
        """
        Filter and select vertical spine regions:
        - Chọn main segment bằng score (area + height)
        - Loại vùng quá xa main segment
        - Chia mask theo trục Y
        - Mỗi đoạn chỉ giữ component lớn nhất
        """

        H, W = binary_mask.shape
        max_dist = max_dist_ratio * W

        # ===============================
        # 1. CONNECTED COMPONENTS
        # ===============================
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            binary_mask, connectivity=8
        )

        regions = []

        for i in range(1, num_labels):
            x, y, w, h, area = stats[i]

            if area < min_area:
                continue

            regions.append({
                "id": i,
                "x1": x,
                "x2": x + w,
                "y1": y,
                "y2": y + h,
                "height": h,
                "area": area
            })

        if not regions:
            return np.zeros_like(binary_mask)

        # ===============================
        # 2. CHỌN MAIN REGION
        # ===============================
        max_area = max(r["area"] for r in regions)
        max_height = max(r["height"] for r in regions)

        for r in regions:
            r["score"] = (
                area_weight * (r["area"] / (max_area + 1e-6)) +
                height_weight * (r["height"] / (max_height + 1e-6))
            )

        main_region = max(regions, key=lambda r: r["score"])

        main_mask = np.zeros_like(binary_mask)
        main_mask[labels == main_region["id"]] = 255

        # ===============================
        # 3. GIỮ REGION GẦN MAIN
        # ===============================
        valid_mask = np.zeros_like(binary_mask)

        for r in regions:
            # region_mask = np.zeros_like(binary_mask)
            # region_mask[labels == r["id"]] = 255

            # dist = self.mask_min_distance_dt(region_mask, main_mask)

            # if dist <= max_dist:
            #     valid_mask[labels == r["id"]] = 255
            valid_mask[labels == r["id"]] = 255


        # ===============================
        # 4. REMOVE CONTAINMENT (NEW LOGIC)
        # ===============================
        result = self.keep_largest_component_per_y_segment(valid_mask, max_dist_ratio)

        return result


    # ==================== LONGEST PATH EXTRACTION ====================

    def longest_path_from_top(self, skel):
        """
        Find longest path from top of skeleton using BFS

        Args:
            skel: Binary skeleton (0/1)

        Returns:
            Mask containing only the longest path
        """
        H, W = skel.shape
        coords = np.column_stack(np.where(skel))

        if len(coords) == 0:
            return skel

        # Select topmost pixel (smallest y)
        start = tuple(coords[np.argmin(coords[:, 0])])

        dist = -np.ones_like(skel, dtype=int)
        parent = {}

        q = deque([start])
        dist[start] = 0
        parent[start] = None

        # BFS to find farthest pixel
        while q:
            cy, cx = q.popleft()
            for dy in [-1, 0, 1]:
                for dx in [-1, 0, 1]:
                    if dy == 0 and dx == 0:
                        continue
                    ny, nx = cy + dy, cx + dx
                    if 0 <= ny < H and 0 <= nx < W:
                        if skel[ny, nx] and dist[ny, nx] == -1:
                            dist[ny, nx] = dist[cy, cx] + 1
                            parent[(ny, nx)] = (cy, cx)
                            q.append((ny, nx))

        # Find farthest pixel
        farthest = max(parent.keys(), key=lambda p: dist[p])

        # Backtrack path
        path = set()
        cur = farthest
        while cur is not None:
            path.add(cur)
            cur = parent[cur]

        out = np.zeros_like(skel)
        for y, x in path:
            out[y, x] = 1

        return out

    def prune_mask_by_longest_path(self, binary):
        """
        Remove extra branches by keeping only longest path from each component

        Args:
            binary: Binary mask (0/255)

        Returns:
            Pruned binary mask
        """
        binary = (binary > 0).astype(np.uint8)

        num_labels, labels = cv2.connectedComponents(binary)

        final_skel = np.zeros_like(binary)

        for label in range(1, num_labels):
            comp = (labels == label).astype(np.uint8)

            # Keep only longest path in component
            main_path = self.longest_path_from_top(comp)

            final_skel |= main_path

        return (final_skel * 255).astype(np.uint8)


    # ==================== CURVE EXTRACTION ====================

    def extract_curve_from_mask(self, binary_mask, num_points=200):
        """
        Extract curve centerline from predicted mask

        Args:
            binary_mask: Binary mask (H, W)
            num_points: Number of points to sample on curve

        Returns:
            curve_points: (N, 2) array of curve coordinates or None
        """

        # Find curve points
        y_coords, x_coords = np.where(binary_mask > 0)

        if len(x_coords) < 3:
            return None

        # Sort by y coordinate
        sorted_idx = np.argsort(y_coords)
        x_sorted = x_coords[sorted_idx]
        y_sorted = y_coords[sorted_idx]

        # Remove duplicates and smooth
        points = np.column_stack([x_sorted, y_sorted])

        # Fit smooth spline
        try:
            curve_points = SpineCurveGenerator.fit_smooth_curve(
                points, num_points=num_points, smoothing=1.0
            )
            return curve_points
        except:
            return points

    def sample_points_by_y(self, curve, num_points=CONFIG['num_point_to_smoth']):
        """
        Sample num_points evenly spaced along Y-axis from top to bottom

        Args:
            curve: (N, 2) array, each row is [x, y]
            num_points: Number of points to sample

        Returns:
            Sampled points (num_points, 2)
        """
        curve_sorted = curve[np.argsort(curve[:, 1])]

        y_values = curve_sorted[:, 1]

        y_samples = np.linspace(y_values[0], y_values[-1], num_points)

        x_samples = np.interp(y_samples, y_values, curve_sorted[:, 0])

        sampled_points = np.vstack([x_samples, y_samples]).T

        return sampled_points

    def smooth_curve_from_points(self, points, method="B-spline", smooth_factor=30,
                                 num_samples=300, window=11, poly=3):
        """
        Smooth curve from points using B-spline or Savitzky-Golay filter

        Args:
            points: List of Nx2 points
            method: "B-spline" or "savgol"
            smooth_factor: Spline smoothness (s). 0 = passes through all points
            num_samples: Number of interpolated spline points
            window, poly: Savitzky-Golay filter parameters

        Returns:
            Smoothed curve points
        """
        points = np.array(points)
        if len(points) < 5:
            raise ValueError("Need at least 5 points for smoothing")

        x = points[:, 0]
        y = points[:, 1]

        if method == "B-spline":
            tck, _ = splprep([x, y], s=smooth_factor)

            unew = np.linspace(0, 1, num_samples)
            out = splev(unew, tck)

            curve = np.vstack(out).T
            return curve

        elif method == "savgol":
            if window % 2 == 0:
                window += 1

            x_smooth = savgol_filter(x, window, poly)
            y_smooth = savgol_filter(y, window, poly)
            return np.vstack([x_smooth, y_smooth]).T

        else:
            raise ValueError("Method must be 'B-spline' or 'savgol'")

    def draw_curve_on_mask(self, curve_points, height, width, thickness=8):
        """
        Draw spline curve on binary mask (0/255)

        Args:
            curve_points: Array of curve points
            height: Mask height
            width: Mask width
            thickness: Line thickness

        Returns:
            Binary mask with drawn curve
        """
        mask = np.zeros((height, width), dtype=np.uint8)

        pts = np.round(curve_points).astype(int)

        cv2.polylines(mask, [pts], isClosed=False, color=255, thickness=thickness)

        return mask

    def sample_points_from_mask(self, mask, num_samples=10):
        """
        Extract num_samples evenly spaced points from mask (mask == 255)

        Args:
            mask: Binary mask
            num_samples: Number of samples to extract

        Returns:
            Array of shape (N, 2) = (x, y)
        """
        # Find all pixels in mask
        ys, xs = np.where(mask == 255)
        coords = np.vstack([xs, ys]).T  # (N, 2)

        if len(coords) == 0:
            return np.empty((0, 2))

        # Sample N evenly spaced points
        num = min(num_samples, len(coords))
        indices = np.linspace(0, len(coords) - 1, num, dtype=int)

        return coords[indices]


    def merge_vertical_paths(self, skel):
        skel = (skel > 0).astype(np.uint8)
        H, W = skel.shape

        merged = np.zeros_like(skel)

        ys, xs = np.where(skel > 0)

        if len(ys) == 0:
            return merged

        from collections import defaultdict
        y_to_xs = defaultdict(list)

        for y, x in zip(ys, xs):
            y_to_xs[y].append(x)

        for y, xs in y_to_xs.items():
            xs = np.array(xs)

            # dùng median thay vì mean (ổn định hơn)
            x_new = int(np.median(xs))
            merged[y, x_new] = 255

        return merged


    def merge_and_connect_continuous(self, skel, thickness=1):
        skel = (skel > 0).astype(np.uint8)
        H, W = skel.shape

        ys, xs = np.where(skel > 0)
        if len(ys) == 0:
            return np.zeros_like(skel)

        from collections import defaultdict
        y_to_xs = defaultdict(list)

        for y, x in zip(ys, xs):
            y_to_xs[y].append(x)

        y_vals = sorted(y_to_xs.keys())

        x_centers = {}
        for y in y_vals:
            x_centers[y] = int(np.median(y_to_xs[y]))

        known_y = np.array(list(x_centers.keys()))
        known_x = np.array(list(x_centers.values()))

        y_min, y_max = known_y.min(), known_y.max()
        full_y = np.arange(y_min, y_max + 1)

        # ---- spline thay vì linear ----
        if len(known_y) >= 4:
            from scipy.interpolate import UnivariateSpline
            spline = UnivariateSpline(known_y, known_x, s=1.5)
            interp_x = spline(full_y)
        else:
            interp_x = np.interp(full_y, known_y, known_x)

        interp_x = np.clip(interp_x, 0, W - 1).astype(int)

        result = np.zeros_like(skel)
        points = np.column_stack([interp_x, full_y]).astype(np.int32)

        cv2.polylines(result, [points], False, 255, thickness)
        return result



    # ==================== ANGLE CALCULATION ====================

    def angle_between_vectors(self, v1, v2):
        """
        Calculate angle between two vectors in degrees

        Args:
            v1, v2: Vectors

        Returns:
            Angle in degrees
        """
        v1_u = v1 / np.linalg.norm(v1)
        v2_u = v2 / np.linalg.norm(v2)

        dot = np.clip(np.dot(v1_u, v2_u), -1.0, 1.0)
        return np.degrees(np.arccos(dot))

    def check_collinearity(self, points, threshold_deg=5):
        """
        Check if points are nearly collinear
        Calculate maximum angle between all pairs of consecutive vectors

        Args:
            points: Array of spine points
            threshold_deg: Threshold for collinearity (not used currently)

        Returns:
            Maximum angle between any pair of vectors
        """
        points = np.array(points)
        vectors = np.diff(points, axis=0)  # (N-1, 2)

        v0 = vectors[0]
        angles_v0 = [self.angle_between_vectors(v0, v) for v in vectors[1:]]

        # Calculate angles between all pairs of vectors
        for i in range(len(vectors)):
            for j in range(i + 1, len(vectors)):
                ang = self.angle_between_vectors(vectors[i], vectors[j])
                self.all_angles.append(ang)

        max_angle_all = np.max(self.all_angles)

        print("Angle between v0 and other vectors:", angles_v0)
        print("Max angle (v0 vs others):", np.max(angles_v0), "degrees\n")

        print("Angles between all pairs of vectors:", self.all_angles)
        print("Max angle between any 2 vectors:", max_angle_all, "degrees")

        return max_angle_all

    def classify_scoliosis(self, cobb_angle):
        """
        Return text description of scoliosis severity based on Cobb angle

        Args:
            cobb_angle: Measured Cobb angle

        Returns:
            Classification string
        """
        if cobb_angle < 10:
            return "0–9°: Not scoliosis (mild postural deviation)"
        elif cobb_angle < 25:
            return "10–24°: Mild. Monitoring & physical therapy"
        elif cobb_angle < 40:
            return "25–39°: Moderate. May need bracing"
        elif cobb_angle < 50:
            return "40–49°: Severe. High risk of progression"
        else:
            return "≥50°: Very severe. Consider surgery"

    # ==================== VISUALIZATION ====================

    def overlay_mask_on_image(self, image, mask, points, color=(255, 105, 97)):
        """
        Overlay mask on image and draw points

        Args:
            image: Base image
            mask: Binary mask
            points: Numpy array of shape (N, 2)
            color: RGB color for mask overlay

        Returns:
            Image with overlay
        """
        image = image.copy()

        # Color the mask
        colored_mask = np.zeros_like(image)
        colored_mask[mask == 255] = color

        overlay = cv2.addWeighted(image, 1.0, colored_mask, 0.7, 0)

        # Draw points if provided
        if points is not None and len(points) > 0:
            for (x, y) in points.astype(int):
                cv2.circle(overlay, (int(x), int(y)), 5, (0, 255, 255), -1)  # Yellow

        return overlay

    def add_rgb_binary(self, processed_image, mask_back):
        mask_back = np.asarray(mask_back)
        H, W = processed_image.shape[:2]

        mask_back = cv2.resize(mask_back, (W, H), interpolation=cv2.INTER_NEAREST)

        # đảm bảo binary
        mask_back = (mask_back > 0).astype(np.float32)

        # đảm bảo image float
        img = processed_image.astype(np.float32)

        img = img * mask_back[:, :, None]

        # trả về đúng dtype ban đầu
        return img.astype(processed_image.dtype)



    def largest_connected_component(self, binary_mask):
        """
        Giữ lại component có diện tích lớn nhất trong binary mask

        Args:
            binary_mask: np.ndarray (H, W), giá trị 0/255 hoặc 0/1

        Returns:
            mask_largest: np.ndarray (H, W), chỉ chứa component lớn nhất
        """

        # đảm bảo mask là uint8 0/255
        mask = (binary_mask > 0).astype(np.uint8) * 255

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )

        if num_labels <= 1:
            return np.zeros_like(mask)

        # bỏ background (label 0)
        areas = stats[1:, cv2.CC_STAT_AREA]
        max_label = 1 + np.argmax(areas)

        result = np.zeros_like(mask)
        result[labels == max_label] = 255

        return result


    def handled_mask(self, mask_pred, mask_back):

        mask_back = np.asarray(mask_back)
        H, W = mask_pred.shape[:2]

        mask_back = cv2.resize(mask_back, (W, H), interpolation=cv2.INTER_NEAREST)


        binary_mask = self.cover_pred_to_binary(mask_pred)

        # handle pre mask
        binary_mask_after = self.filter_vertical_regions(binary_mask)

        binary_mask_after = self._skeletonize_with_skeletonize(binary_mask_after)

        # binary_mask_after = self._skeletonize_with_skeletonize(binary_mask)

        binary_mask_after = self.prune_mask_by_longest_path(binary_mask_after)

        binary_mask_after = self._skeletonize_with_skeletonize(binary_mask_after)

        binary_mask_after = self.merge_vertical_paths(binary_mask_after)

        binary_mask_after = self.merge_and_connect_continuous(binary_mask_after)

        binary_mask_after= binary_mask_after * mask_back

        # binary_mask_after = self.smooth_from_skeleton(binary_mask_after)

        # binary_mask_after = self._skeletonize_with_skeletonize(binary_mask_after)


        # take curve point from mask
        curve_points = self.extract_curve_from_mask(binary_mask_after)

        return curve_points, binary_mask, binary_mask_after



    # ==================== PREDICTION ====================

    def predict(self, image_rgb, image_size=512):
        """
        Predict spine curve from image

        Args:
            image_rgb: RGB image
            image_size: Size for model input

        Returns:
            Tuple of (mask_pred, curve_points, binary_mask, binary_mask_after)
        """
        orig_h, orig_w = image_rgb.shape[:2]

        # Preprocess
        transform = A.Compose([
            A.Resize(image_size, image_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

        img_tensor = transform(image=image_rgb)['image'].unsqueeze(0).to(self.device)

        # Predict
        with torch.no_grad():
            pred, deep_sup_preds = self.model(img_tensor)
            pred = torch.sigmoid(pred)

        # Convert to numpy
        mask_pred = pred[0, 0].cpu().numpy()

        # Resize to original size
        mask_pred_resized = cv2.resize(mask_pred, (orig_w, orig_h))

        # Convert mask_pred to binary mask


        return mask_pred_resized

    def predict_with_tta(self, image_rgb, image_size=512):
        """
        Predict with Test-Time Augmentation

        Args:
            image_rgb: RGB image
            image_size: Size for model input

        Returns:
            Tuple of (mask_pred, curve_points, binary_mask, binary_mask_after)
        """
        orig_h, orig_w = image_rgb.shape[:2]

        # TTA transforms
        tta_transforms = [
            A.Compose([
                A.Resize(image_size, image_size),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ]),
            A.Compose([
                A.Resize(image_size, image_size),
                A.HorizontalFlip(p=1.0),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ]),
            A.Compose([
                A.Resize(int(image_size * 1.1), int(image_size * 1.1)),
                A.CenterCrop(image_size, image_size),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ]),
        ]

        predictions = []

        for transform in tta_transforms:
            img_tensor = transform(image=image_rgb)['image'].unsqueeze(0).to(self.device)

            with torch.no_grad():
                pred, deep_sup_preds = self.model(img_tensor)
                pred = torch.sigmoid(pred)

            predictions.append(pred[0, 0].cpu().numpy())

        # Average predictions
        mask_pred = np.mean(predictions, axis=0)
        mask_pred_resized = cv2.resize(mask_pred, (orig_w, orig_h))

        # Convert mask_pred to binary mask

        return mask_pred_resized

    def visualize(self, image_path: str, save_path: str = None, use_tta=False,
                  saveFunction=KeypointSaver(output_dir="/content/output"),
                  isSaveResults=False, model_segmentation_back=None,config=None,
                  BodyLandmarkDetector=None):
        """
        Visualize prediction results

        Args:
            image_path: Path to input image
            save_path: Path to save visualization
            use_tta: Whether to use test-time augmentation
            saveFunction: Function to save keypoints
            isSaveResults: Whether to save results
            model_segmentation_back: Background segmentation model
            BodyLandmarkDetector: Body landmark detector (not used)

        Returns:
            Tuple of (mask_pred, curve_points)
        """
        # Load original image
        self.all_angles = []
        self.pre = PreProcessing()
        image = cv2.imread(str(image_path))
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        processed_image = self.pre.process(str(image_path))
        h, w = processed_image.shape[:2]

        mask_back = model_segmentation_back.predict(image_rgb=processed_image)

        mask_back = self.largest_connected_component(mask_back)

        processed_image = self.add_rgb_binary(processed_image, mask_back)

        # Predict
        if use_tta:
            mask_pred = self.predict_with_tta(processed_image)
        else:
            mask_pred = self.predict(processed_image)

        curve_points, binary_mask, binary_mask_after = self.handled_mask(mask_pred, mask_back)



        # Take n points from curve_points
        curve_points_n = self.sample_points_by_y(curve_points, num_points=16)

        # Smooth curve from N points
        curve_points = self.smooth_curve_from_points(curve_points_n, method="B-spline", smooth_factor=35, num_samples=16)

        curve_mask = self.draw_curve_on_mask(curve_points, h, w, thickness=2)


        # processed, resultVisualize, stats = process_and_visualize(
        #     raw_points=curve_points,
        #     background_img=processed_image,
        #     show_raw=True,
        #     verbose=True,
        #     spine_color=(0, 120, 255),
        #     joint_color=(255, 100, 0),
        #     thickness=3
        # )

        resultVisualize, processed = spine_visual(curve_points, processed_image)

        # Take num points according to mask
        point_according_to_mask = self.sample_points_from_mask(curve_mask, num_samples=CONFIG['num_point_to_smoth'])

        overlay = self.overlay_mask_on_image(processed_image, curve_mask, point_according_to_mask)

        # Create visualization
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        axes = axes.flatten()  # Convert 2D to 1D for axes[0..7]

        # 0 — Original image
        axes[0].imshow(processed_image)
        axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
        axes[0].axis('off')

        # 1 — Predicted mask
        axes[1].imshow(processed_image)
        mask_colored = np.zeros((h, w, 3), dtype=np.uint8)
        mask_colored[:, :, 1] = (mask_pred * 255).astype(np.uint8)
        axes[1].imshow(mask_colored, alpha=0.5)
        axes[1].set_title('Predicted Mask', fontsize=14, fontweight='bold')
        axes[1].axis('off')

        # 2 — Extracted curve
        axes[2].imshow(processed_image)
        if curve_points is not None and len(curve_points) > 0:
            axes[2].plot(curve_points[:, 0], curve_points[:, 1], 'c-', linewidth=3)
            axes[2].scatter(curve_points[::20, 0], curve_points[::20, 1],
                          c='yellow', s=50, edgecolors='white', linewidths=2)
        axes[2].set_title('Extracted Spine Curve', fontsize=14, fontweight='bold')
        axes[2].axis('off')

        # # 3 — Binary Back
        # axes[3].imshow(mask_back)
        # axes[3].set_title('Binary Back', fontsize=14, fontweight='bold')
        # axes[3].axis('off')

        # 4 — Binary Spline
        axes[3].imshow(binary_mask)
        axes[3].set_title('Binary Spline', fontsize=14, fontweight='bold')
        axes[3].axis('off')

        # 5 —Overlay spline (after smoothing N points)
        axes[4].imshow(binary_mask_after)
        axes[4].set_title('Smooth Overlay', fontsize=14, fontweight='bold')
        axes[4].axis('off')

        # 6 — Curve mask
        axes[5].imshow(resultVisualize)
        axes[5].set_title('curve_mask', fontsize=14, fontweight='bold')
        axes[5].axis('off')

        # 7 — Final overlay
        axes[6].imshow(overlay)
        axes[6].set_title('Final Overlay', fontsize=14, fontweight='bold')
        axes[6].axis('off')

        # Calculate collinearity and classify scoliosis
        result = self.check_collinearity(point_according_to_mask, threshold_deg=5)

        fig.text(
            0.5, 0.02,
            f"""Collinearity Result: {self.classify_scoliosis(result)}""",
            ha='center', fontsize=14, color="red"
        )

        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"✓ Visualization saved to {save_path}")

        plt.show()
        plt.pause(0.1)

        print("\n" + "="*80)
        if isSaveResults:
            self.confirm_and_run(saveFunction.save_image_with_keypoints, image_rgb,
                               point_according_to_mask, self.random_filename("jpg"))
            if os.path.exists(image_path):
                os.remove(image_path)

        return mask_pred, curve_points




# ============================================================================
# EVALUATION METRICS
# ============================================================================

# def compute_metrics(pred_mask, gt_mask, threshold=0.5):
#     """Compute evaluation metrics"""
#     pred_binary = (pred_mask > threshold).astype(np.float32)
#     gt_binary = gt_mask.astype(np.float32)

#     # Dice coefficient
#     intersection = (pred_binary * gt_binary).sum()
#     dice = (2.0 * intersection) / (pred_binary.sum() + gt_binary.sum() + 1e-6)

#     # IoU
#     union = ((pred_binary + gt_binary) > 0).sum()
#     iou = intersection / (union + 1e-6)

#     # Precision & Recall
#     tp = intersection
#     fp = pred_binary.sum() - tp
#     fn = gt_binary.sum() - tp

#     precision = tp / (tp + fp + 1e-6)
#     recall = tp / (tp + fn + 1e-6)
#     f1 = 2 * precision * recall / (precision + recall + 1e-6)

#     return {
#         'dice': dice,
#         'iou': iou,
#         'precision': precision,
#         'recall': recall,
#         'f1': f1
#     }


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == '__main__':

    # ========== AUTO-GENERATE PATHS ==========
    # Create output directories if they don't exist
    output_dir = Path(CONFIG['output_dir'])
    output_dir.mkdir(exist_ok=True, parents=True)

    checkpoint_dir = Path(CONFIG['checkpoint_dir'])
    checkpoint_dir.mkdir(exist_ok=True, parents=True)

    visualization_dir = Path(CONFIG['visualization_dir'])
    visualization_dir.mkdir(exist_ok=True, parents=True)

    batch_pred_dir = Path(CONFIG['batch_prediction_dir'])
    batch_pred_dir.mkdir(exist_ok=True, parents=True)

    logs_dir = Path(CONFIG['logs_dir'])
    logs_dir.mkdir(exist_ok=True, parents=True)

    # Generate run name if not provided
    if CONFIG['run_name'] is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        CONFIG['run_name'] = f"{CONFIG['experiment_name']}_{timestamp}"

    # ========== SET RANDOM SEED ==========
    if CONFIG['seed'] is not None:
        torch.manual_seed(CONFIG['seed'])
        np.random.seed(CONFIG['seed'])
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(CONFIG['seed'])

        if CONFIG['deterministic']:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        else:
            torch.backends.cudnn.benchmark = CONFIG['benchmark']

    # ========== PRINT CONFIGURATION ==========
    print("CONFIGURATION SUMMARY")
    print("="*80)




    config = SpineConfig()
    config.N = 16

    # ========== DATASET PREPARATION ==========
    # Initialize augmentation scheduler
    aug_scheduler = DynamicAugmentationScheduler(
        image_size=CONFIG ['image_size']
    )


    # # Create datasets
    # train_dataset = SpineCurveDataset(
    #     image_dir=CONFIG['train_images'],
    #     coco_file=CONFIG['train_annotations'],
    #     augmentation_scheduler=aug_scheduler,
    #     is_train=True,
    #     curve_thickness=CONFIG['curve_thickness'],
    #     num_curve_points=CONFIG['num_curve_points']
    # )


    # train_dataset = SpineCurveDataset(
    #     image_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images',
    #     coco_file='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json',
    #     augmentation_scheduler=aug_scheduler,
    #     is_train=True,
    #     curve_thickness=CONFIG['curve_thickness'],
    #     num_curve_points=CONFIG['num_curve_points']
    # )

    train_dataset = SpineCurveDataset(
        image_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images',
        coco_file='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json',
        augmentation_scheduler=aug_scheduler,
        is_train=True,
        brightness_limit=0.3,      # ±30% độ sáng
        contrast_limit=0.3,        # ±30% độ tương phản
        rotation_limit=15          # ±15 độ xoay
    )

    val_dataset = SpineCurveDataset(
        image_dir=CONFIG['val_images'],
        coco_file=CONFIG['val_annotations'],
        augmentation_scheduler=aug_scheduler,
        is_train=False,
        curve_thickness=CONFIG['curve_thickness'],
        num_curve_points=CONFIG['num_curve_points']
    )

    test_dataset = SpineCurveDataset(
        image_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/test/Images',
        coco_file='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/test/_annotations.coco.json',
        augmentation_scheduler=aug_scheduler,
        is_train=False,
        curve_thickness=CONFIG['curve_thickness'],
        num_curve_points=CONFIG['num_curve_points']
    )

    # train_dataset = SpineCurveDataset(
    #     image_dir='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images',
    #     coco_file='/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json',
    #     augmentation_scheduler=aug_scheduler,
    #     is_train=True,
    #     curve_thickness=CONFIG['curve_thickness'],
    #     num_curve_points=CONFIG['num_curve_points']
    # )



    # val_dataset = SpineCurveDataset(
    #     image_dir=CONFIG['val_images'],
    #     coco_file=CONFIG['val_annotations'],
    #     augmentation_scheduler=aug_scheduler,
    #     is_train=False,
    #     curve_thickness=CONFIG['curve_thickness'],
    #     num_curve_points=CONFIG['num_curve_points']
    # )


    print(f"\n✓ Datasets created:")
    print(f"  - Training: {len(train_dataset)} images")
    print(f"  - Validation: {len(val_dataset)} images")




    # ========== MODEL INITIALIZATION ==========

    print("\n🤖 Initializing model...")
    # model = SpineCurveDetector(backbone='efficientnet_b2')

    config = ModelConfig()
    config.spine_vertical_kernel = (13, 1)
    config.spine_att_scales = [3, 5, 7, 9]

    model = SpineCurveDetectorV2(config)

    # model = SpineCurveDetector(backbone="tf_efficientnetv2_m")



    model_segmentation_back = BackSegmentationModel(
        backbone='resnext50_32x4d',
        encoder_weights='imagenet',
        device='cuda'
    )

    model_segmentation_back.load_model(
        "/content/drive/MyDrive/unet/coco_v3.1_unet++/unetpp_back_epoch_50.pth"
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"  - Total parameters: {total_params:,}")
    print(f"  - Trainable parameters: {trainable_params:,}")



    # ========== TRAINING ==========
    print("\n" + "="*80)
    print("STARTING TRAINING")
    print("="*80)



    # train_spine_curve_model(
    #     model=model,
    #     train_dataset=train_dataset,
    #     val_dataset=val_dataset,
    #     device='cuda',
    #     epochs=CONFIG['epochs'],
    #     batch_size=4,
    #     lr=CONFIG['lr'],
    #     save_path=CONFIG['model_save_path'],
    #     plot_save_path='/content/drive/MyDrive/RESULTS/plot.png',
    #     # model_segmentation_back=model_segmentation_back,
    #     #resume_from=CONFIG['model_save_path']  #path model save for load checkpoint

    #     gradient_accumulation_steps=4,  # Cao
    #     clear_cache_every_n_steps=10,  # Clear thường xuyên
    #     clear_cache_after_backward=True,


    #     #Multi-scale config
    #     use_multiscale_val=False,  # Tắt mặc định
    #     multiscale_scales=[0.75, 1.0, 1.5],   # Chỉ dùng 1 scale
    #     multiscale_flip=True,     # Tắt flip
    # )


    evaluate_on_test_set(
        model=model,
        test_dataset=test_dataset,
        device='cuda',
        batch_size=4,
        checkpoint_path=CONFIG['model_save_path'],
        use_multiscale=False,
        multiscale_scales=[0.75, 1.0, 1.5],
        multiscale_flip=True,
        accuracy_thresholds=[0.05, 0.1, 0.15],
        save_predictions=False,
        predictions_dir='test_predictions',
        num_visualize=5
    )


    # ========== INFERENCE ==========
    print("\n" + "="*80)
    print("TESTING INFERENCE")
    print("="*80)

    predictor = SpineCurvePredictor(
        model_path=CONFIG['model_save_path'],
        device='cuda',
        threshold=0.6
    )


    # ========== BATCH TESTING ==========
    print("\n" + "="*80)
    print("BATCH TESTING ON VALIDATION SET")
    print("="*80)


    # mask_pred, curve_points = predictor.visualize(
    #     '/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/test/Images/Nhn_bit_sm_cong_vo_ct_sng_va_cach_phong_nga_resize.jpg',
    #     save_path='',
    #     use_tta=True,
    #     isSaveResults=False,
    #     model_segmentation_back=model_segmentation_back,
    #     config=config,
    #     #BodyLandmarkDetector=SpineHipKeypointDetector
    # )

    from pathlib import Path
    val_image_dir = Path(CONFIG['test_images'])
    test_images = list(val_image_dir.glob('*.jpg'))[:90]

    output_dir = Path(CONFIG['batch_prediction_dir'])
    output_dir.mkdir(exist_ok=True, parents=True)

    saver = KeypointSaver(output_dir="/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/Images", annotation_file="/content/drive/MyDrive/DATA/scoliosis2.v3i.coco/train_aug/_annotations.coco.json")

    for i, img_path in enumerate(test_images, 1):
        print(f"\n[{i}/{len(test_images)}] Processing: {img_path.name}")

        output_path = output_dir / f"pred_{img_path.stem}.jpg"

        try:
            mask_pred, curve_points = predictor.visualize(
                str(img_path),
                save_path=str(output_path),
                use_tta=True,
                saveFunction=saver,
                isSaveResults=False,
                model_segmentation_back=model_segmentation_back,
                config=config,
                #BodyLandmarkDetector=SpineHipKeypointDetector
            )

            if curve_points is not None:
                print(f"  ✓ Curve detected ({len(curve_points)} points)")
            else:
                print(f"  ⚠ No curve detected")

        except Exception as e:
            print(f"  ✗ Error: {str(e)}")

    print(f"📈 Model saved to: {CONFIG['model_save_path']}")

    # ========== SUMMARY ==========
    print("\n" + "="*80)
    print("TRAINING SUMMARY")
    print("="*80)
    print(f"""
    Model Architecture:
      - Backbone: EfficientNet-B2
      - Decoder: U-Net style with skip connections
      - Loss: Dice + Focal + Deep Supervision
        """)

Output hidden; open in https://colab.research.google.com to view.